# Analytics Build: Tourism Activity, Recovery, and Economic Alignment

## Purpose

This notebook builds the Analytics layer for ProvInsights PH Tourism from Gold-layer inputs. It produces five persisted Delta tables in `tuklas_dev`.`05_analytics` that answer the project's business questions (BQ1–BQ2) and their analytical questions (AQ).

## Prerequisites

Gold tables must exist and be populated:

- `tuklas_dev`.`04_gold`.`dim_province`: 100 reporting units
- `tuklas_dev`.`04_gold`.`dim_year`: 2019–2024
- `tuklas_dev`.`04_gold`.`fact_province_year`: 600 rows (100 units × 6 years)

The Gold notebooks (`00_gold_setup`, `01_gold_dim_province`, `02_gold_dim_year`, `03_gold_fact_province_year`) must have completed successfully.

## Outputs

Five persisted Delta tables in `tuklas_dev`.`05_analytics`:

| Table | Grain | Key | BQ/AQ |
| --- | --- | --- | --- |
| `tourism_activity` | reporting unit × year | (`province_psgc`, `year`) | BQ1-AQ1.1 |
| `tourism_concentration` | year | (`year`) | BQ1-AQ1.1 |
| `tourism_recovery` | reporting unit × comparison year | (`province_psgc`, `comparison_year`) | BQ1-AQ1.2 |
| `traveler_segment_recovery` | reporting unit × comparison year | (`province_psgc`, `comparison_year`) | BQ1-AQ1.3 |
| `tourism_economy` | reporting unit | (`province_psgc`) | BQ2-AQ2.1, BQ2-AQ2.2, BQ2-AQ2.3 |

## Business questions and analytical questions

**BQ1:** Which provinces stand out in tourism activity and recovery, and which appear to be lagging?

- **AQ1.1:** Highest/lowest activity and intensity, and concentration → `tourism_activity` + `tourism_concentration`
- **AQ1.2:** Recovery relative to 2019 → `tourism_recovery`
- **AQ1.3:** Traveler-type recovery among units below 2019 → `traveler_segment_recovery`

**BQ2:** How is recovery in tourism activity associated with changes in local tourism-related economic activity across provinces?

- **AQ2.1:** Traveler change versus real A&F GVA growth
- **AQ2.2:** Matching versus divergent directions
- **AQ2.3:** A&F economic significance across recovery patterns → `tourism_economy`

Additional context: 2023 tourism intensity versus 2023 GDP per capita.

## Execution order

**A** Overview → **B** Schema setup → **C** Gold input validation → **D** Tourism activity and concentration → **E** Tourism recovery → **F** Traveler-segment recovery → **G** Tourism and economic alignment → **H** Final validation and manual spot checks → **I** Execution summary

Each phase builds candidate results in a temporary view, runs blocking checks, and only then replaces the persisted table. If a blocking check fails, execution stops before the table is replaced.

## Rerun behavior

All tables use `CREATE OR REPLACE TABLE`, so the notebook is idempotent. Rerunning replaces each table in sequence. Replacement of separate Delta tables is **not** one atomic transaction: if execution stops after some tables were replaced, the completion summary (Phase I) reports which outputs completed and which did not.

## Conventions

- SQL cells are used for all analytical transformations (DL-005, DL-009).
- A descriptive Markdown cell precedes every executable cell.
- Missing values are never replaced with zero (DL-009).
- Values are not rounded before classification.
- Every denominator is guarded against NULL and nonpositive values.
- Base values from Gold are preserved alongside derived metrics.
- Reporting-unit identifiers and labels are carried through.
- Comparability and interpretation flags from Gold are preserved.
- Metric-specific eligibility and exclusion reasons are explicit.
- Findings describe distributions, changes, and associations: not causation (DL-008).
- `above_2019` is not described as proof of an "emerging destination"; the baseline-leader comparison and emergence rules remain proposed until approved.

In [0]:
%python
from datetime import datetime
import uuid

CATALOG = "tuklas_dev"
GOLD_SCHEMA = "04_gold"
ANALYTICS_SCHEMA = "05_analytics"
DQ_SCHEMA = "06_data_quality"
NOTEBOOK_NAME = "01_build_analytics"
run_id = f"analytics_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

BASELINE_YEAR = 2019
RECOVERY_YEARS = [2022, 2023]
COMPARISON_YEAR = 2023
ALL_YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

# --- Delta version snapshot for consistent Gold reads ---
# Gold may change between validation and writing because temp views are lazy.
# Record the version at start so VERSION AS OF can be used for consistency.
gold_versions = {}
for tbl in ['fact_province_year', 'dim_province', 'dim_year']:
    try:
        v = spark.sql(f"SELECT version FROM (DESCRIBE HISTORY `{CATALOG}`.`{GOLD_SCHEMA}`.`{tbl}` LIMIT 1)").collect()[0][0]
        gold_versions[tbl] = v
    except Exception:
        gold_versions[tbl] = None
gold_fact_version = gold_versions.get('fact_province_year')
gold_dim_province_version = gold_versions.get('dim_province')
gold_dim_year_version = gold_versions.get('dim_year')

# --- Tracking structures ---
tables_completed = []    # table_name
tables_blocked = []      # table_name
run_log = []             # (table_name, status, row_count, version, timestamp, notes)

# Missing Gold fields discovered during validation -- reported, never invented
missing_gold_fields = []
# Unresolved decisions that affect this notebook
unresolved_decisions = []

# Validation results: (check_id, table, severity, status, failed_rows, message, phase, checked_at)
val_results = []
review_flag_count = 0

def val_check(check_id, table, failed_rows, message="", severity="BLOCK", phase=""):
    """Record a validation check. severity = BLOCK or REVIEW.
    BLOCK failures stop execution; REVIEW failures are reported but do not stop.
    failed_rows = actual count of failed rows (never replaced with 0)."""
    global review_flag_count
    status = "PASS" if failed_rows == 0 else "FAIL"
    checked_at = datetime.now().isoformat()
    val_results.append((check_id, table, severity, status, int(failed_rows), message, phase, checked_at))
    marker = "\u2713" if status == "PASS" else "\u2717"
    sev_tag = f" ({severity})" if severity != "BLOCK" else ""
    print(f"{marker} {check_id:<14} {table:<28} {status:<5}{sev_tag} | {message}")
    if severity == "REVIEW" and failed_rows > 0:
        review_flag_count += 1

def assert_blocking(phase):
    """Raise an error if any BLOCK-severity check failed. Only BLOCK failures stop execution.
    REVIEW failures remain visible and contribute to review_flag_count."""
    blocks = [(r[0], r[4], r[5]) for r in val_results if r[2] == "BLOCK" and r[3] == "FAIL"]
    if blocks:
        msg = "; ".join(f"{b[0]}: {b[1]} failures -- {b[2]}" for b in blocks)
        raise RuntimeError(f"BLOCKING checks failed in {phase}: {msg}")
    review_in_phase = sum(1 for r in val_results if r[2] == "REVIEW" and r[3] == "FAIL")
    if review_in_phase:
        print(f"  -> All {phase} BLOCK checks passed. {review_in_phase} REVIEW flag(s) reported above.")
    else:
        print(f"  -> All {phase} blocking checks passed.")

def record_output(table_name, row_count, validation_status="PASS"):
    """Record a successfully persisted output with row count, validation status, and table version."""
    completed_at = datetime.now().isoformat()
    try:
        tv = spark.sql(f"SELECT version FROM (DESCRIBE HISTORY `{CATALOG}`.`{ANALYTICS_SCHEMA}`.`{table_name}` LIMIT 1)").collect()[0][0]
    except Exception:
        tv = None
    tables_completed.append(table_name)
    run_log.append((table_name, "COMPLETED", row_count, tv, completed_at, f"validation: {validation_status}"))

def record_failure(table_name, error_msg):
    """Record a persistence failure. Reports which outputs were already replaced."""
    tables_blocked.append(table_name)
    run_log.append((table_name, "FAILED", None, None, datetime.now().isoformat(), error_msg[:200]))

def persist_dq_results():
    """Persist all DQ check results to the control table."""
    if not val_results:
        return
    from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
    rows = [(run_id, NOTEBOOK_NAME, r[0], r[1], r[2], r[3], r[4], None, None, None, datetime.now(), r[5]) for r in val_results]
    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("notebook_name", StringType(), True),
        StructField("check_id", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("severity", StringType(), True),
        StructField("status", StringType(), True),
        StructField("failed_rows", LongType(), True),
        StructField("expected", StringType(), True),
        StructField("observed", StringType(), True),
        StructField("sample_keys", StringType(), True),
        StructField("checked_at", TimestampType(), True),
        StructField("notes", StringType(), True),
    ])
    spark.createDataFrame(rows, schema).write.mode("append").saveAsTable(f"`{CATALOG}`.`{DQ_SCHEMA}`.`dq_results`")

# Report missing Gold fields (DL-024 references derived_values; DL-011 references per_capita_recomputed)
missing_gold_fields.append(("foreign_includes_overseas_filipinos", "traveler_segment_recovery -- foreign/overseas Filipino separate-segment analysis blocked"))
missing_gold_fields.append(("derived_values", "tourism_economy -- economic derivation flags from DL-024"))
missing_gold_fields.append(("per_capita_recomputed", "tourism_economy -- per-capita recomputation flag from DL-011"))

# Unresolved decisions (DL-019 is Proposed, not Accepted)
unresolved_decisions.append(("DL-019", "reported_partial policy is Proposed, not Accepted. Partial-reporting units are included as eligible with status preserved as caveat, but this treatment is not final."))

print(f"Run ID: {run_id}")
print(f"Catalog: {CATALOG}")
print(f"Analytics schema: {ANALYTICS_SCHEMA}")
print(f"Gold fact_province_year version: {gold_fact_version}")
print(f"Gold dim_province version: {gold_dim_province_version}")
print(f"Gold dim_year version: {gold_dim_year_version}")
print(f"Baseline year: {BASELINE_YEAR}")
print(f"Recovery years: {RECOVERY_YEARS}")
print(f"Comparison year for economy: {COMPARISON_YEAR}")
print(f"Missing Gold fields: {len(missing_gold_fields)}")
for f, m in missing_gold_fields:
    print(f"  - {f}: {m}")
print(f"Unresolved decisions: {len(unresolved_decisions)}")
for d, m in unresolved_decisions:
    print(f"  - {d}: {m}")

## Phase B: Schema setup

**Purpose:** Create the destination schema `tuklas_dev`.`05_analytics` so it is ready to receive the five Analytics tables.

**Supported BQ/AQ:** All (infrastructure for BQ1-BQ2).

**Input:** None (DDL only).

**Output:** Schema `tuklas_dev`.`05_analytics` exists.

**Rerun:** `CREATE SCHEMA IF NOT EXISTS` is idempotent and safe to rerun.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `tuklas_dev`.`05_analytics`

## Phase C: Gold input validation

**Purpose:** Verify that the three Gold source tables exist, have the expected row counts, and contain all columns required by the Analytics layer. These are blocking checks: if any fails, the notebook stops before building any Analytics table.

**Supported BQ/AQ:** All (guards every downstream metric).

**Inputs:**
- `tuklas_dev`.`04_gold`.`dim_province`: expected 100 rows, unique `province_psgc`
- `tuklas_dev`.`04_gold`.`dim_year`: expected 6 rows (2019–2024)
- `tuklas_dev`.`04_gold`.`fact_province_year`: expected 600 rows (100 × 6)

**Output:** Temp views `v_val_dim_province`, `v_val_dim_year`, `v_val_fact` with check results. If any check fails, the Python assertion raises an error and execution stops.

**Failure consequence:** If Gold tables are missing, have wrong row counts, or lack required columns, no Analytics table is built. The error message identifies which check failed.

### C1: dim_province validation

This cell checks `dim_province` for:
- 100 rows (one per reporting unit)
- `province_psgc` is unique, non-NULL, and 10 digits
- `province_name`, `unit_type`, `region_psgc`, `region_name` are non-NULL

**Input:** `tuklas_dev`.`04_gold`.`dim_province`

**Output:** Temp view `v_val_c1` with one row: the count of failing rows (0 = pass).

**Failure consequence:** BLOCK: the notebook cannot build any Analytics table without a valid province dimension.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_c1 AS
SELECT
  -- Row count must be 100
  CASE WHEN COUNT(*) != 100 THEN 1 ELSE 0 END
  + -- province_psgc unique
  CASE WHEN COUNT(DISTINCT province_psgc) != COUNT(province_psgc) THEN 1 ELSE 0 END
  + -- province_psgc non-NULL
  CASE WHEN COUNT(*) - COUNT(province_psgc) > 0 THEN 1 ELSE 0 END
  + -- province_psgc 10 digits
  CASE WHEN SUM(CASE WHEN province_psgc NOT RLIKE '^[0-9]{10}$' THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + -- province_name non-NULL and non-blank
  CASE WHEN COUNT(*) - COUNT(province_name) > 0 OR SUM(CASE WHEN TRIM(province_name) = '' THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + -- unit_type non-NULL and non-blank
  CASE WHEN COUNT(*) - COUNT(unit_type) > 0 OR SUM(CASE WHEN TRIM(unit_type) = '' THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + -- region_psgc non-NULL and non-blank
  CASE WHEN COUNT(*) - COUNT(region_psgc) > 0 OR SUM(CASE WHEN TRIM(region_psgc) = '' THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + -- region_name non-NULL and non-blank
  CASE WHEN COUNT(*) - COUNT(region_name) > 0 OR SUM(CASE WHEN TRIM(region_name) = '' THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  AS failed_checks
FROM `tuklas_dev`.`04_gold`.`dim_province`

### C2: dim_year validation

This cell checks `dim_year` for:
- Exact year set: 2019–2024 (6 rows)
- `year` is unique and non-NULL
- Baseline flag: `is_baseline_year` TRUE only for 2019, non-NULL
- Recovery flag: `is_recovery_year` TRUE only for 2022 and 2023, non-NULL
- Comparison-window flag: `in_comparison_window` TRUE only for 2019–2023, non-NULL
- All required flags are non-NULL

**Input:** `tuklas_dev`.`04_gold`.`dim_year`

**Output:** Temp view `v_val_c2` with the count of failing checks.

**Failure consequence:** BLOCK: recovery and economy tables depend on correct year flags.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_c2 AS
SELECT
  -- Exact year set must be 2019-2024
  CASE WHEN SORT_ARRAY(COLLECT_LIST(year)) != ARRAY(2019, 2020, 2021, 2022, 2023, 2024) THEN 1 ELSE 0 END
  + -- year unique and non-NULL
  CASE WHEN COUNT(DISTINCT year) != COUNT(year) OR COUNT(*) - COUNT(year) > 0 THEN 1 ELSE 0 END
  + -- Exactly 6 rows
  CASE WHEN COUNT(*) != 6 THEN 1 ELSE 0 END
  + -- Baseline flag TRUE only for 2019
  CASE WHEN SUM(CASE WHEN is_baseline_year AND year != 2019 THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CAST(is_baseline_year AS INT)) != 1 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN is_baseline_year IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + -- Recovery flag TRUE only for 2022 and 2023
  CASE WHEN SORT_ARRAY(COLLECT_LIST(CASE WHEN is_recovery_year THEN year END)) != ARRAY(2022, 2023) THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN is_recovery_year IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + -- Comparison-window flag TRUE only for 2019-2023
  CASE WHEN SORT_ARRAY(COLLECT_LIST(CASE WHEN in_comparison_window THEN year END)) != ARRAY(2019, 2020, 2021, 2022, 2023) THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN in_comparison_window IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  AS failed_checks
FROM `tuklas_dev`.`04_gold`.`dim_year`

### C3: fact_province_year validation

This cell checks `fact_province_year` for:
- 600 rows (100 units × 6 years)
- All `province_psgc` values exist in `dim_province`
- All `year` values exist in `dim_year`
- Required columns are present: `total_travelers`, `domestic_travelers`, `foreign_travelers`, `overseas_filipinos`, `traveler_value_status`, `population_total`, `population_source`, `comparable_to_2019`, `gdp_total_current`, `afs_gva_current`, `afs_gva_constant`, `gdp_per_capita_current`

**Input:** `tuklas_dev`.`04_gold`.`fact_province_year` joined to `dim_province` and `dim_year`.

**Output:** Temp view `v_val_c3` with the count of failing checks.

**Failure consequence:** BLOCK: if the fact table is incomplete or missing required columns, affected metrics are reported as blocked.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_c3 AS
WITH fact AS (
  SELECT f.*
  FROM `tuklas_dev`.`04_gold`.`fact_province_year` f
)
SELECT
  -- Row count must be 600
  CASE WHEN COUNT(*) != 600 THEN 1 ELSE 0 END
  + -- province_psgc non-NULL
  CASE WHEN COUNT(*) - COUNT(f.province_psgc) > 0 THEN 1 ELSE 0 END
  + -- year non-NULL
  CASE WHEN COUNT(*) - COUNT(f.year) > 0 THEN 1 ELSE 0 END
  + -- (province_psgc, year) unique
  CASE WHEN COUNT(*) != COUNT(DISTINCT CONCAT(f.province_psgc, '_', CAST(f.year AS STRING))) THEN 1 ELSE 0 END
  + -- All province_psgc exist in dim_province
  CASE WHEN COUNT(f.province_psgc) - COUNT(p.province_psgc) > 0 THEN 1 ELSE 0 END
  + -- All years exist in dim_year
  CASE WHEN COUNT(f.year) - COUNT(d.year) > 0 THEN 1 ELSE 0 END
  AS failed_checks
FROM fact f
LEFT JOIN `tuklas_dev`.`04_gold`.`dim_province` p ON f.province_psgc = p.province_psgc
LEFT JOIN `tuklas_dev`.`04_gold`.`dim_year` d ON f.year = d.year

### C4: Required column availability

This cell checks that every column the Analytics layer needs exists in `fact_province_year`. If a required field is missing, the affected metric is reported as blocked: no workaround is invented and no metric is marked validated.

**Input:** `tuklas_dev`.`04_gold`.`fact_province_year` (schema only).

**Output:** Temp view `v_val_c4` listing any missing columns and the metrics they affect.

**Failure consequence:** BLOCK: the error message names each missing column and the metric it blocks.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_c4 AS
WITH required AS (
  SELECT 'total_travelers' AS col, 'tourism_activity, tourism_recovery, tourism_economy' AS affected_metrics, 'BLOCK' AS severity
  UNION ALL SELECT 'domestic_travelers', 'traveler_segment_recovery', 'BLOCK'
  UNION ALL SELECT 'foreign_travelers', 'traveler_segment_recovery', 'BLOCK'
  UNION ALL SELECT 'overseas_filipinos', 'traveler_segment_recovery', 'BLOCK'
  UNION ALL SELECT 'traveler_value_status', 'tourism_activity, tourism_recovery, traveler_segment_recovery', 'BLOCK'
  UNION ALL SELECT 'population_total', 'tourism_activity, tourism_economy', 'BLOCK'
  UNION ALL SELECT 'population_source', 'tourism_activity', 'BLOCK'
  UNION ALL SELECT 'comparable_to_2019', 'tourism_recovery, traveler_segment_recovery, tourism_economy', 'BLOCK'
  UNION ALL SELECT 'gdp_total_current', 'tourism_economy', 'BLOCK'
  UNION ALL SELECT 'gdp_per_capita_current', 'tourism_economy', 'BLOCK'
  UNION ALL SELECT 'afs_gva_current', 'tourism_economy', 'BLOCK'
  UNION ALL SELECT 'afs_gva_constant', 'tourism_economy', 'BLOCK'
  -- Interpretation/caveat fields -- missing from Gold, reported not invented
  UNION ALL SELECT 'foreign_includes_overseas_filipinos', 'traveler_segment_recovery -- separate foreign/overseas Filipino analysis', 'REVIEW'
  UNION ALL SELECT 'derived_values', 'tourism_economy -- economic derivation flags (DL-024)', 'REVIEW'
  UNION ALL SELECT 'per_capita_recomputed', 'tourism_economy -- per-capita recomputation flag (DL-011)', 'REVIEW'
),
actual AS (
  SELECT column_name FROM `tuklas_dev`.INFORMATION_SCHEMA.COLUMNS
  WHERE table_schema = '04_gold' AND table_name = 'fact_province_year'
)
SELECT r.col, r.affected_metrics, r.severity
FROM required r
LEFT JOIN actual a ON r.col = a.column_name
WHERE a.column_name IS NULL

### C5: Assert Gold validation

This cell collects the results of checks C1–C4 and raises an error if any BLOCK-severity check failed. Missing BLOCK-severity columns are reported with the affected metrics. Missing REVIEW-severity fields (e.g. `foreign_includes_overseas_filipinos`, `derived_values`, `per_capita_recomputed`) are reported as REVIEW flags: they do not stop execution but the affected analysis is documented as blocked or limited.

**Failure consequence:** BLOCK: execution stops if any BLOCK-severity column is missing. No Analytics table is built.

In [0]:
%python
c1 = spark.table("v_val_c1").collect()[0][0]
val_check("C1", "dim_province", c1, "100 unique rows, valid PKs, non-NULL/non-blank labels", phase="Phase C")

c2 = spark.table("v_val_c2").collect()[0][0]
val_check("C2", "dim_year", c2, "6 years, baseline=2019, recovery=2022-2023, comparison window=2019-2023, all flags non-NULL", phase="Phase C")

c3 = spark.table("v_val_c3").collect()[0][0]
val_check("C3", "fact_province_year", c3, "600 rows, FK integrity", phase="Phase C")

missing_rows = spark.table("v_val_c4").collect()
block_missing = [r for r in missing_rows if r.severity == 'BLOCK']
review_missing = [r for r in missing_rows if r.severity == 'REVIEW']

if block_missing:
    details = "; ".join(f"{r.col} blocks {r.affected_metrics}" for r in block_missing)
    val_check("C4", "fact_province_year", len(block_missing), f"Missing BLOCK columns: {details}", phase="Phase C")
else:
    val_check("C4", "fact_province_year", 0, "All required BLOCK columns present", phase="Phase C")

if review_missing:
    details = "; ".join(f"{r.col} ({r.affected_metrics})" for r in review_missing)
    val_check("C4b", "fact_province_year", len(review_missing), f"Missing REVIEW fields (reported, not invented): {details}", severity="REVIEW", phase="Phase C")

assert_blocking("Phase C -- Gold input validation")

## Phase D: Tourism activity and concentration

**Purpose:** Build two tables that answer BQ1-AQ1.1 -- *Highest/lowest activity and intensity, and concentration.*

**Supported BQ/AQ:**
- BQ1-AQ1.1: Highest/lowest activity and intensity, and concentration -> `tourism_activity` + `tourism_concentration`

This phase produces two persisted tables:
- `tourism_activity` (grain: reporting unit × year): volume, intensity, ranks, and share
- `tourism_concentration` (grain: year): top-5 and top-10 concentration of eligible travelers

**Inputs:** `dim_province`, `dim_year`, `fact_province_year` (Gold only; no Bronze or Silver).

### D1: Build `tourism_activity` candidate

This cell creates a temporary view `v_tourism_activity` with one row per reporting unit × year (600 rows). It joins `fact_province_year` to `dim_province` and computes:

- **Activity eligibility:** `total_travelers` is non-NULL, nonnegative, and `traveler_value_status` is a known reported status (not `not_reported`, not NULL). A genuine reported zero is eligible. NULL or unknown reporting statuses do not qualify silently.
- **Intensity eligibility:** activity-eligible AND `population_total` > 0. The denominator is guarded against NULL and nonpositive values.
- **`tourism_intensity_per_1000`** = `total_travelers` / `population_total` × 1000. NULL when ineligible. Not rounded.
- **`tourism_rank`** = descending `RANK()` over eligible values in each year, allowing ties. NULL for ineligible rows.
- **`intensity_rank`** = descending `RANK()` over intensity-eligible values in each year. NULL for ineligible rows.
- **`tourism_share_pct`** = unit's `total_travelers` / sum of eligible travelers in that year × 100. NULL for ineligible rows.
- **Partial reporting policy:** `reported_partial` units are included as eligible because they have reported values. The `traveler_value_status` is preserved as a caveat. However, the `reported_partial` treatment is based on DL-019, which is **Proposed** (not Accepted) in the decision log. This policy dependency is reported in the execution summary.
- **Reporting break against 2019:** a `comparable_to_2019` value of FALSE does not exclude valid single-year activity. The flag is preserved for downstream comparison use.

**Input:** `tuklas_dev`.`04_gold`.`fact_province_year` joined to `tuklas_dev`.`04_gold`.`dim_province`.

**Output:** Temp view `v_tourism_activity`.

**Failure consequence:** If the temp view cannot be created, no `tourism_activity` or `tourism_concentration` table is built.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_tourism_activity AS
WITH base AS (
  SELECT
    p.province_psgc,
    p.province_name,
    p.unit_type,
    p.region_psgc,
    p.region_name,
    p.comparability_note,
    f.year,
    f.total_travelers,
    f.population_total,
    f.population_source,
    f.traveler_value_status,
    f.comparable_to_2019,
    CASE WHEN f.total_travelers IS NULL THEN FALSE
         WHEN f.total_travelers < 0 THEN FALSE
         WHEN f.traveler_value_status IS NULL THEN FALSE
         WHEN f.traveler_value_status = 'not_reported' THEN FALSE
         ELSE TRUE END AS activity_eligible,
    CASE WHEN f.total_travelers IS NULL THEN 'total_travelers is NULL'
         WHEN f.total_travelers < 0 THEN 'total_travelers is negative'
         WHEN f.traveler_value_status IS NULL THEN 'traveler_value_status is NULL'
         WHEN f.traveler_value_status = 'not_reported' THEN 'traveler_value_status is not_reported'
         ELSE NULL END AS activity_exclusion_reason,
    CASE WHEN f.total_travelers IS NULL THEN FALSE
         WHEN f.total_travelers < 0 THEN FALSE
         WHEN f.traveler_value_status IS NULL THEN FALSE
         WHEN f.traveler_value_status = 'not_reported' THEN FALSE
         WHEN f.population_total IS NULL THEN FALSE
         WHEN f.population_total <= 0 THEN FALSE
         ELSE TRUE END AS intensity_eligible,
    CASE WHEN f.total_travelers IS NULL THEN 'total_travelers is NULL'
         WHEN f.total_travelers < 0 THEN 'total_travelers is negative'
         WHEN f.traveler_value_status IS NULL THEN 'traveler_value_status is NULL'
         WHEN f.traveler_value_status = 'not_reported' THEN 'traveler_value_status is not_reported'
         WHEN f.population_total IS NULL THEN 'population_total is NULL'
         WHEN f.population_total <= 0 THEN 'population_total is nonpositive'
         ELSE NULL END AS intensity_exclusion_reason,
    CASE WHEN f.total_travelers IS NOT NULL AND f.total_travelers >= 0
           AND f.traveler_value_status IS NOT NULL AND f.traveler_value_status != 'not_reported'
           AND f.population_total IS NOT NULL AND f.population_total > 0
    THEN CAST(f.total_travelers AS DOUBLE) / CAST(f.population_total AS DOUBLE) * 1000.0
    ELSE NULL END AS tourism_intensity_per_1000,
    CASE WHEN f.total_travelers IS NOT NULL AND f.total_travelers >= 0
           AND f.traveler_value_status IS NOT NULL AND f.traveler_value_status != 'not_reported'
    THEN f.total_travelers ELSE NULL END AS tourism_rank_sort
  FROM `tuklas_dev`.`04_gold`.`fact_province_year` f
  JOIN `tuklas_dev`.`04_gold`.`dim_province` p ON f.province_psgc = p.province_psgc
),
ranked AS (
  SELECT
    b.*,
    CASE WHEN b.activity_eligible
      THEN RANK() OVER (PARTITION BY b.year ORDER BY b.tourism_rank_sort DESC NULLS LAST)
      ELSE NULL END AS tourism_rank,
    CASE WHEN b.intensity_eligible
      THEN RANK() OVER (PARTITION BY b.year ORDER BY b.tourism_intensity_per_1000 DESC NULLS LAST)
      ELSE NULL END AS intensity_rank,
    SUM(CASE WHEN b.activity_eligible THEN b.total_travelers ELSE 0 END) OVER (PARTITION BY b.year) AS eligible_total_year
  FROM base b
)
SELECT
  province_psgc, province_name, unit_type, region_psgc, region_name,
  year, total_travelers, population_total, population_source,
  tourism_intensity_per_1000,
  tourism_rank,
  intensity_rank,
  CASE WHEN activity_eligible AND eligible_total_year > 0
    THEN CAST(total_travelers AS DOUBLE) / CAST(eligible_total_year AS DOUBLE) * 100.0
    ELSE NULL END AS tourism_share_pct,
  traveler_value_status,
  comparable_to_2019,
  comparability_note,
  activity_eligible,
  activity_exclusion_reason,
  intensity_eligible,
  intensity_exclusion_reason
FROM ranked
ORDER BY year, tourism_rank_sort DESC NULLS LAST, province_psgc

### D2: Validate `tourism_activity`

This cell runs six blocking checks on the candidate view:

| Check | What it validates | Failure consequence |
| --- | --- | --- |
| D1 | Natural key (`province_psgc`, `year`) is unique and non-NULL | BLOCK: duplicate or missing keys corrupt all downstream tables |
| D2 | Every eligible row has non-NULL, nonnegative `total_travelers` | BLOCK: eligibility logic is wrong |
| D3 | `tourism_intensity_per_1000` is non-NULL only when `population_total` > 0 | BLOCK: denominator not guarded |
| D4 | `tourism_rank` and `intensity_rank` are NULL for ineligible rows | BLOCK: ranks leaked to ineligible rows |
| D5 | `tourism_share_pct` sums to approximately 100% per year (tolerance: 0.01 percentage points) | BLOCK: share denominator is wrong |

**Numerical tolerance:** D5 uses a tolerance of 0.01 percentage points to account for floating-point rounding.

**Input:** Temp view `v_tourism_activity`.

**Output:** Temp view `v_val_d_activity` with one row per check (check_id, failures).

**Failure consequence:** BLOCK: the Python assertion in D3 raises an error and the `tourism_activity` table is not replaced.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_d_activity AS
WITH checks AS (
  SELECT 'D1_dup_keys' AS check_id, COUNT(*) AS failures FROM (
    SELECT province_psgc, year FROM v_tourism_activity GROUP BY province_psgc, year HAVING COUNT(*) > 1
  )
  UNION ALL
  SELECT 'D1_null_keys', COUNT(*) FROM v_tourism_activity WHERE province_psgc IS NULL OR year IS NULL
  UNION ALL
  SELECT 'D2_bad_eligible', COUNT(*) FROM v_tourism_activity WHERE activity_eligible AND (total_travelers IS NULL OR total_travelers < 0)
  UNION ALL
  SELECT 'D3_bad_intensity', COUNT(*) FROM v_tourism_activity WHERE tourism_intensity_per_1000 IS NOT NULL AND (population_total IS NULL OR population_total <= 0)
  UNION ALL
  SELECT 'D4_rank_ineligible', COUNT(*) FROM v_tourism_activity WHERE NOT activity_eligible AND tourism_rank IS NOT NULL
  UNION ALL
  SELECT 'D4_intensity_rank_ineligible', COUNT(*) FROM v_tourism_activity WHERE NOT intensity_eligible AND intensity_rank IS NOT NULL
  UNION ALL
  SELECT 'D5_share_not_100', COUNT(*) FROM (
    SELECT year, SUM(tourism_share_pct) AS s
    FROM v_tourism_activity WHERE activity_eligible
    GROUP BY year
    HAVING ABS(s - 100.0) > 0.01
  )
)
SELECT * FROM checks

### D3: Assert `tourism_activity` checks and persist

This cell collects the check results from D2. If any check has failures > 0, it raises an error and the table is **not** replaced. If all checks pass, the SQL cell that follows replaces the persisted table.

**Input:** Temp view `v_val_d_activity`.

**Output:** `tuklas_dev`.`05_analytics`.`tourism_activity` is replaced (if all checks pass).

**Rerun:** `CREATE OR REPLACE TABLE` is idempotent.

In [0]:
%python
for row in spark.table("v_val_d_activity").collect():
    val_check(row.check_id, "v_tourism_activity", row.failures,
              f"{row.check_id}: {row.failures} failures", phase="Phase D")
assert_blocking("Phase D1 -- tourism_activity validation")
record_output("tourism_activity", spark.table("v_tourism_activity").count())

### D4: Persist `tourism_activity`

This cell replaces the persisted Delta table `tuklas_dev`.`05_analytics`.`tourism_activity` with the validated candidate view. It uses `CREATE OR REPLACE TABLE ... USING DELTA AS SELECT ...` for reproducible table replacement.

**Input:** Temp view `v_tourism_activity` (validated by D2–D3).

**Output:** Persisted Delta table `tuklas_dev`.`05_analytics`.`tourism_activity` with grain (province_psgc, year) and 600 rows.

**Rerun:** Idempotent: replaces the table on each run.

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`tourism_activity`
USING DELTA
AS SELECT
  province_psgc,
  province_name,
  unit_type,
  region_psgc,
  region_name,
  year,
  total_travelers,
  population_total,
  population_source,
  tourism_intensity_per_1000,
  tourism_rank,
  intensity_rank,
  tourism_share_pct,
  traveler_value_status,
  comparable_to_2019,
  comparability_note,
  activity_eligible,
  activity_exclusion_reason,
  intensity_eligible,
  intensity_exclusion_reason
FROM v_tourism_activity

### D5: Build `tourism_concentration` candidate

This cell creates a temporary view `v_tourism_concentration` with one row per year. It starts from the complete configured year set (2019–2024) and LEFT JOINs eligible aggregates. Years with no eligible units remain present with zero metadata counts and NULL shares.

**Deterministic tie-break rule:** units are ranked by `total_travelers DESC, province_psgc ASC`.

**Denominator:** the sum of eligible reporting-unit travelers in that year.

**Formulas:**
- `top_5_share_pct` = sum of top-5 travelers / eligible total x 100
- `top_10_share_pct` = sum of top-10 travelers / eligible total x 100
- If the denominator is zero or unavailable, shares are NULL.
- If fewer than 5 (or 10) eligible units exist, all available units are used and their count is exposed in `top_5_unit_count` (or `top_10_unit_count`).
- A year with no eligible units has `eligible_reporting_unit_count = 0`, `excluded_reporting_unit_count = total expected units`, `top_5_unit_count = 0`, `top_10_unit_count = 0`, and NULL shares. Zero metadata counts do not mean missing traveler values have been converted to zero.

**Input:** Temp view `v_tourism_activity` (eligible rows only).

**Output:** Temp view `v_tourism_concentration`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_tourism_concentration AS
WITH all_years AS (
  SELECT explode(array(2019, 2020, 2021, 2022, 2023, 2024)) AS year
),
total_units AS (
  SELECT year, COUNT(*) AS total_units FROM v_tourism_activity GROUP BY year
),
eligible AS (
  SELECT year, province_psgc, total_travelers
  FROM v_tourism_activity
  WHERE activity_eligible
),
ranked AS (
  SELECT
    year, province_psgc, total_travelers,
    ROW_NUMBER() OVER (PARTITION BY year ORDER BY total_travelers DESC, province_psgc ASC) AS rn
  FROM eligible
),
year_agg AS (
  SELECT
    year,
    COUNT(*) AS eligible_reporting_unit_count,
    SUM(total_travelers) AS eligible_total_travelers
  FROM eligible GROUP BY year
),
top5 AS (
  SELECT year, COUNT(*) AS top_5_unit_count, SUM(total_travelers) AS top_5_travelers
  FROM ranked WHERE rn <= 5 GROUP BY year
),
top10 AS (
  SELECT year, COUNT(*) AS top_10_unit_count, SUM(total_travelers) AS top_10_travelers
  FROM ranked WHERE rn <= 10 GROUP BY year
)
SELECT
  ay.year,
  COALESCE(ya.eligible_reporting_unit_count, 0) AS eligible_reporting_unit_count,
  COALESCE(tu.total_units, 0) - COALESCE(ya.eligible_reporting_unit_count, 0) AS excluded_reporting_unit_count,
  ya.eligible_total_travelers,
  t5.top_5_travelers,
  COALESCE(t5.top_5_unit_count, 0) AS top_5_unit_count,
  CASE WHEN ya.eligible_total_travelers IS NOT NULL AND ya.eligible_total_travelers > 0
    THEN CAST(t5.top_5_travelers AS DOUBLE) / CAST(ya.eligible_total_travelers AS DOUBLE) * 100.0
    ELSE NULL END AS top_5_share_pct,
  t10.top_10_travelers,
  COALESCE(t10.top_10_unit_count, 0) AS top_10_unit_count,
  CASE WHEN ya.eligible_total_travelers IS NOT NULL AND ya.eligible_total_travelers > 0
    THEN CAST(t10.top_10_travelers AS DOUBLE) / CAST(ya.eligible_total_travelers AS DOUBLE) * 100.0
    ELSE NULL END AS top_10_share_pct,
  'Included reporting-unit total (eligible units only); not national coverage' AS coverage_note
FROM all_years ay
LEFT JOIN total_units tu ON ay.year = tu.year
LEFT JOIN year_agg ya ON ay.year = ya.year
LEFT JOIN top5 t5 ON ay.year = t5.year
LEFT JOIN top10 t10 ON ay.year = t10.year
ORDER BY ay.year

### D6: Validate `tourism_concentration`

This cell runs four blocking checks:

| Check | What it validates | Failure consequence |
| --- | --- | --- |
| D6 | One row per year, with no NULL years | BLOCK: grain is wrong |
| D7 | `eligible_reporting_unit_count` + `excluded_reporting_unit_count` = total units per year (100) | BLOCK: eligible/excluded counts don't reconcile |
| D8 | `top_5_share_pct` ≤ `top_10_share_pct` ≤ 100% when denominator is positive | BLOCK: share logic is wrong |
| D9 | Concentration denominator matches sum of eligible travelers in `v_tourism_activity` per year | BLOCK: denominator mismatch |

**Tolerance:** D8 uses a tolerance of 0.001 percentage points for floating-point comparison.

**Input:** Temp view `v_tourism_concentration` and `v_tourism_activity`.

**Output:** Temp view `v_val_d_concentration`.

**Failure consequence:** BLOCK: the `tourism_concentration` table is not replaced.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_d_concentration AS
WITH checks AS (
  SELECT 'D6_dup_years' AS check_id, COUNT(*) AS failures FROM (
    SELECT year FROM v_tourism_concentration GROUP BY year HAVING COUNT(*) > 1
  )
  UNION ALL
  SELECT 'D6_null_years', COUNT(*) FROM v_tourism_concentration WHERE year IS NULL
  UNION ALL
  SELECT 'D6_year_coverage', CASE
    WHEN SORT_ARRAY(COLLECT_LIST(year)) != ARRAY(2019, 2020, 2021, 2022, 2023, 2024) THEN 1
    ELSE 0 END
  FROM v_tourism_concentration
  UNION ALL
  SELECT 'D7_count_mismatch', COUNT(*) FROM v_tourism_concentration
  WHERE eligible_reporting_unit_count + excluded_reporting_unit_count !=
    (SELECT COUNT(*) FROM v_tourism_activity a WHERE a.year = v_tourism_concentration.year)
  UNION ALL
  SELECT 'D7_count_reconcile', COUNT(*) FROM v_tourism_concentration c
  JOIN (SELECT year, COUNT(*) AS total_units FROM v_tourism_activity GROUP BY year) a ON c.year = a.year
  WHERE c.eligible_reporting_unit_count + c.excluded_reporting_unit_count != a.total_units
  UNION ALL
  SELECT 'D8_share_violation', COUNT(*) FROM v_tourism_concentration
  WHERE eligible_total_travelers IS NOT NULL AND eligible_total_travelers > 0
    AND (top_5_share_pct > top_10_share_pct + 0.001 OR top_10_share_pct > 100.001)
  UNION ALL
  SELECT 'D8_null_shares_no_eligible', COUNT(*) FROM v_tourism_concentration
  WHERE eligible_reporting_unit_count = 0 AND (top_5_share_pct IS NOT NULL OR top_10_share_pct IS NOT NULL)
  UNION ALL
  SELECT 'D9_denominator_mismatch', COUNT(*) FROM (
    SELECT c.year, c.eligible_total_travelers AS conc_total,
      SUM(CASE WHEN a.activity_eligible THEN a.total_travelers ELSE 0 END) AS activity_total
    FROM v_tourism_concentration c
    LEFT JOIN v_tourism_activity a ON c.year = a.year
    GROUP BY c.year, c.eligible_total_travelers
    HAVING conc_total IS NOT NULL AND conc_total != activity_total
  )
)
SELECT * FROM checks

### D7: Assert `tourism_concentration` checks and persist

This cell collects the check results from D6. If any check has failures > 0, it raises an error and the table is **not** replaced. If all checks pass, the SQL cell that follows replaces the persisted table.

**Input:** Temp view `v_val_d_concentration`.

**Output:** `tuklas_dev`.`05_analytics`.`tourism_concentration` is replaced (if all checks pass).

In [0]:
%python
for row in spark.table("v_val_d_concentration").collect():
    val_check(row.check_id, "v_tourism_concentration", row.failures,
              f"{row.check_id}: {row.failures} failures", phase="Phase D")
assert_blocking("Phase D2 -- tourism_concentration validation")
record_output("tourism_concentration", spark.table("v_tourism_concentration").count())

### D8: Persist `tourism_concentration`

This cell replaces the persisted Delta table `tuklas_dev`.`05_analytics`.`tourism_concentration` with the validated candidate view.

**Input:** Temp view `v_tourism_concentration` (validated by D6–D7).

**Output:** Persisted Delta table with grain (year) and one row per year.

**Rerun:** Idempotent: replaces the table on each run.

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`tourism_concentration`
USING DELTA
AS SELECT
  year,
  eligible_reporting_unit_count,
  excluded_reporting_unit_count,
  eligible_total_travelers,
  top_5_travelers,
  top_5_unit_count,
  top_5_share_pct,
  top_10_travelers,
  top_10_unit_count,
  top_10_share_pct,
  coverage_note
FROM v_tourism_concentration

## Phase E: Tourism recovery

**Purpose:** Build the `tourism_recovery` table that answers BQ1-AQ1.2 -- *Recovery relative to 2019.*

**Supported BQ/AQ:**
- BQ1-AQ1.2: Recovery relative to 2019 -> `tourism_recovery`

**Baseline:** 2019 (DL-004).
**Comparison years:** 2022 and 2023, calculated separately.

Note: `above_2019` is computed but is NOT described as proof of an "emerging destination." The baseline-leader comparison and emergence rules remain proposed until approved (DL-006, OI-7).

**Formula:** `recovery_index` = `travelers_comparison_year` / `travelers_2019` × 100

**Eligibility rules:**
- Baseline travelers > 0.
- Comparison travelers exist and are nonnegative.
- Comparison-year `comparable_to_2019` is explicitly TRUE. FALSE or NULL does not qualify.
- Approved coverage rules satisfied (implicitly: non-NULL travelers means the unit was reported).

**Classification (unrounded):**
- < 100: `below_2019`
- = 100: `unchanged`
- > 100: `above_2019`
- Ineligible: NULL

Ineligible rows are preserved with NULL recovery metrics and an exclusion reason.

**Input:** `tuklas_dev`.`04_gold`.`fact_province_year` (years 2019, 2022, 2023) joined to `dim_province`.

### E1: Build `tourism_recovery` candidate

This cell creates a temporary view `v_tourism_recovery` with one row per reporting unit × comparison year (200 rows: 100 units × 2 years). It:
- Joins the 2019 baseline to each comparison year (2022, 2023) on `province_psgc`.
- Computes `traveler_change_absolute` = `travelers_comparison_year` − `travelers_2019`.
- Computes `recovery_index` = `travelers_comparison_year` / `travelers_2019` × 100, guarded by eligibility.
- Classifies `recovery_class` using the unrounded `recovery_index`.
- Preserves baseline and comparison `traveler_value_status`, `comparable_to_2019`, and `comparability_note` as caveats.
- Ineligible rows are kept with NULL metrics and an exclusion reason.

**Input:** `fact_province_year` (2019, 2022, 2023) and `dim_province`.

**Output:** Temp view `v_tourism_recovery`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_tourism_recovery AS
WITH baseline AS (
  SELECT province_psgc, total_travelers AS travelers_2019, traveler_value_status AS baseline_traveler_value_status
  FROM `tuklas_dev`.`04_gold`.`fact_province_year` WHERE year = 2019
),
comparison AS (
  SELECT province_psgc, year AS comparison_year, total_travelers AS travelers_comparison_year,
    traveler_value_status AS comparison_traveler_value_status, comparable_to_2019 AS comparison_comparable_to_2019
  FROM `tuklas_dev`.`04_gold`.`fact_province_year` WHERE year IN (2022, 2023)
),
joined AS (
  SELECT
    p.province_psgc, p.province_name, p.unit_type, p.region_psgc, p.region_name, p.comparability_note,
    c.comparison_year, b.travelers_2019, c.travelers_comparison_year,
    b.baseline_traveler_value_status, c.comparison_traveler_value_status,
    c.comparison_comparable_to_2019 AS comparable_to_2019,
    CASE WHEN b.travelers_2019 IS NULL OR b.travelers_2019 <= 0 THEN FALSE
         WHEN c.travelers_comparison_year IS NULL OR c.travelers_comparison_year < 0 THEN FALSE
         WHEN c.comparison_comparable_to_2019 IS NOT TRUE THEN FALSE
         ELSE TRUE END AS recovery_eligible,
    CASE WHEN b.travelers_2019 IS NULL THEN 'baseline travelers_2019 is NULL'
         WHEN b.travelers_2019 <= 0 THEN 'baseline travelers_2019 is nonpositive'
         WHEN c.travelers_comparison_year IS NULL THEN 'comparison travelers is NULL'
         WHEN c.travelers_comparison_year < 0 THEN 'comparison travelers is negative'
         WHEN c.comparison_comparable_to_2019 IS NOT TRUE THEN 'comparison year not comparable to 2019'
         ELSE NULL END AS recovery_exclusion_reason
  FROM baseline b
  JOIN comparison c ON b.province_psgc = c.province_psgc
  JOIN `tuklas_dev`.`04_gold`.`dim_province` p ON b.province_psgc = p.province_psgc
)
SELECT
  province_psgc, province_name, unit_type, region_psgc, region_name,
  comparison_year, travelers_2019, travelers_comparison_year,
  CASE WHEN recovery_eligible THEN travelers_comparison_year - travelers_2019 ELSE NULL END AS traveler_change_absolute,
  CASE WHEN recovery_eligible THEN CAST(travelers_comparison_year AS DOUBLE) / CAST(travelers_2019 AS DOUBLE) * 100.0 ELSE NULL END AS recovery_index,
  CASE WHEN NOT recovery_eligible THEN NULL
       WHEN CAST(travelers_comparison_year AS DOUBLE) / CAST(travelers_2019 AS DOUBLE) * 100.0 < 100 THEN 'below_2019'
       WHEN CAST(travelers_comparison_year AS DOUBLE) / CAST(travelers_2019 AS DOUBLE) * 100.0 = 100 THEN 'unchanged'
       ELSE 'above_2019' END AS recovery_class,
  baseline_traveler_value_status, comparison_traveler_value_status, comparable_to_2019,
  recovery_eligible, recovery_exclusion_reason, comparability_note
FROM joined
ORDER BY province_name, comparison_year

### E2: Validate `tourism_recovery`

This cell runs six blocking checks:

| Check | What it validates | Failure consequence |
| --- | --- | --- |
| E1 | Natural key (`province_psgc`, `comparison_year`) is unique and non-NULL | BLOCK: grain is wrong |
| E2 | Only comparison years 2022 and 2023 are present | BLOCK: wrong comparison years |
| E3 | Expected 200 rows (100 units × 2 years) | BLOCK: missing or extra rows |
| E4 | No eligible row has baseline ≤ 0 or NULL | BLOCK: eligibility logic is wrong |
| E5 | No eligible row has `comparable_to_2019` ≠ TRUE | BLOCK: comparability not checked |
| E6 | Ineligible rows have NULL `recovery_index` and `recovery_class` | BLOCK: metrics leaked to ineligible rows |

**Input:** Temp view `v_tourism_recovery`.

**Output:** Temp view `v_val_e_recovery`.

**Failure consequence:** BLOCK: the `tourism_recovery` table is not replaced.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_e_recovery AS
WITH checks AS (
  SELECT 'E1_dup_keys' AS check_id, COUNT(*) AS failures FROM (
    SELECT province_psgc, comparison_year FROM v_tourism_recovery GROUP BY province_psgc, comparison_year HAVING COUNT(*) > 1
  )
  UNION ALL
  SELECT 'E1_null_keys', COUNT(*) FROM v_tourism_recovery WHERE province_psgc IS NULL OR comparison_year IS NULL
  UNION ALL
  SELECT 'E2_bad_years', COUNT(*) FROM v_tourism_recovery WHERE comparison_year NOT IN (2022, 2023)
  UNION ALL
  SELECT 'E3_row_count', CASE WHEN COUNT(*) != 200 THEN 1 ELSE 0 END FROM v_tourism_recovery
  UNION ALL
  SELECT 'E4_bad_baseline', COUNT(*) FROM v_tourism_recovery WHERE recovery_eligible AND (travelers_2019 IS NULL OR travelers_2019 <= 0)
  UNION ALL
  SELECT 'E5_bad_comparable', COUNT(*) FROM v_tourism_recovery WHERE recovery_eligible AND comparable_to_2019 IS NOT TRUE
  UNION ALL
  SELECT 'E6_metrics_ineligible', COUNT(*) FROM v_tourism_recovery WHERE NOT recovery_eligible AND (recovery_index IS NOT NULL OR recovery_class IS NOT NULL)
)
SELECT * FROM checks

### E3: Assert `tourism_recovery` checks and persist

This cell collects the check results from E2. If any check has failures > 0, it raises an error and the table is **not** replaced.

**Input:** Temp view `v_val_e_recovery`.

**Output:** `tuklas_dev`.`05_analytics`.`tourism_recovery` is replaced (if all checks pass).

In [0]:
%python
for row in spark.table("v_val_e_recovery").collect():
    val_check(row.check_id, "v_tourism_recovery", row.failures,
              f"{row.check_id}: {row.failures} failures", phase="Phase E")
assert_blocking("Phase E -- tourism_recovery validation")
record_output("tourism_recovery", spark.table("v_tourism_recovery").count())

### E4: Persist `tourism_recovery`

This cell replaces the persisted Delta table `tuklas_dev`.`05_analytics`.`tourism_recovery` with the validated candidate view.

**Input:** Temp view `v_tourism_recovery` (validated by E2–E3).

**Output:** Persisted Delta table with grain (province_psgc, comparison_year) and 200 rows.

**Rerun:** Idempotent: replaces the table on each run.

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`tourism_recovery`
USING DELTA
AS SELECT
  province_psgc, province_name, unit_type, region_psgc, region_name,
  comparison_year, travelers_2019, travelers_comparison_year,
  traveler_change_absolute, recovery_index, recovery_class,
  baseline_traveler_value_status, comparison_traveler_value_status,
  comparable_to_2019, recovery_eligible, recovery_exclusion_reason,
  comparability_note
FROM v_tourism_recovery

## Phase F: Traveler-segment recovery

**Purpose:** Build `traveler_segment_recovery` that answers BQ2-AQ3: *Did domestic or foreign travelers recover faster, and where?*

**Supported BQ/AQ:**
- BQ2-AQ3: Did domestic or foreign travelers recover faster, and where?

This table applies recovery rules **independently** to each traveler segment:
- `domestic_travelers`
- `foreign_travelers`
- `overseas_filipinos` (never abbreviated as 'OFW'; always 'overseas Filipino')

**Eligibility (per segment):**
- Segment baseline > 0.
- Segment comparison value is non-NULL and nonnegative.
- Comparison-year `comparable_to_2019` is explicitly TRUE.

**Category caveats:**
- The `foreign_includes_overseas_filipinos` interpretation field is **not available in Gold**. Non-NULL foreign values do not prove that foreign and overseas Filipino counts are separately comparable. Without this field, separate foreign and overseas Filipino segment recovery is **blocked**. Source values are preserved; affected recovery metrics are NULL; eligibility flags are FALSE; explicit exclusion reasons are provided.
- Domestic recovery is calculated independently and is not affected by the foreign/overseas Filipino block.
- The previous blanket statement that Region II reports overseas Filipinos inside Foreign is not supported by the current approved source assessment. Category interpretations are not assumed from region name alone.
- `traveler_value_status` is preserved for both years as a reporting caveat.

**Additional columns:**
- `overall_recovery_index` and `overall_recovery_class`: from `total_travelers`, same rules as `tourism_recovery`.
- `below_2019_total_flag`: TRUE when overall recovery is eligible and below 2019.
- `segment_analysis_cohort_flag`: TRUE for the below-2019 cohort; all rows are kept so other units remain traceable.

**Input:** `fact_province_year` (2019, 2022, 2023) and `dim_province`.

### F1: Build `traveler_segment_recovery` candidate

This cell creates a temporary view `v_traveler_segment_recovery` with one row per reporting unit x comparison year (200 rows). It:
- Joins 2019 baseline to 2022 and 2023 comparison data on `province_psgc`.
- Calculates `recovery_index`, `recovery_class`, `recovery_eligible`, and `exclusion_reason` for the **overall total** and for **domestic** segment recovery (same eligibility rules as `tourism_recovery`: baseline > 0, comparison non-NULL nonnegative, `comparable_to_2019` IS TRUE).
- **Foreign and overseas Filipino segment recovery is BLOCKED**: `foreign_includes_overseas_filipinos` is not available in Gold. Non-NULL foreign values do not prove separate category comparability. Source values are preserved; recovery metrics are NULL; eligibility is FALSE; explicit exclusion reasons are provided.
- Preserves `traveler_value_status` for both years and `comparability_note` as caveats.
- Adds `foreign_segment_supported`, `overseas_filipino_segment_supported`, and `category_block_reason` columns.
- Flags the below-2019 cohort.
- Does not invent or assume category interpretations from region name alone.

**Input:** `fact_province_year` (2019, 2022, 2023) and `dim_province`.

**Output:** Temp view `v_traveler_segment_recovery`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_traveler_segment_recovery AS
WITH baseline AS (
  SELECT province_psgc,
    total_travelers AS total_2019,
    domestic_travelers AS domestic_travelers_2019,
    foreign_travelers AS foreign_travelers_2019,
    overseas_filipinos AS overseas_filipinos_2019,
    traveler_value_status AS baseline_traveler_value_status
  FROM `tuklas_dev`.`04_gold`.`fact_province_year` WHERE year = 2019
),
comparison AS (
  SELECT province_psgc, year AS comparison_year,
    total_travelers AS total_comparison,
    domestic_travelers AS domestic_travelers_comparison,
    foreign_travelers AS foreign_travelers_comparison,
    overseas_filipinos AS overseas_filipinos_comparison,
    traveler_value_status AS comparison_traveler_value_status,
    comparable_to_2019 AS comparison_comparable_to_2019
  FROM `tuklas_dev`.`04_gold`.`fact_province_year` WHERE year IN (2022, 2023)
),
joined AS (
  SELECT
    p.province_psgc, p.province_name, p.unit_type, p.region_psgc, p.region_name, p.comparability_note,
    c.comparison_year,
    b.total_2019, c.total_comparison,
    b.domestic_travelers_2019, c.domestic_travelers_comparison,
    b.foreign_travelers_2019, c.foreign_travelers_comparison,
    b.overseas_filipinos_2019, c.overseas_filipinos_comparison,
    b.baseline_traveler_value_status, c.comparison_traveler_value_status,
    c.comparison_comparable_to_2019 AS comparable_to_2019
  FROM baseline b
  JOIN comparison c ON b.province_psgc = c.province_psgc
  JOIN `tuklas_dev`.`04_gold`.`dim_province` p ON b.province_psgc = p.province_psgc
)
SELECT
  province_psgc, province_name, unit_type, region_psgc, region_name, comparison_year,
  -- Overall recovery
  CASE WHEN total_2019 IS NOT NULL AND total_2019 > 0 AND total_comparison IS NOT NULL AND total_comparison >= 0 AND comparable_to_2019 IS TRUE
    THEN CAST(total_comparison AS DOUBLE) / CAST(total_2019 AS DOUBLE) * 100.0 ELSE NULL END AS overall_recovery_index,
  CASE WHEN total_2019 IS NULL OR total_2019 <= 0 OR total_comparison IS NULL OR total_comparison < 0 OR comparable_to_2019 IS NOT TRUE THEN NULL
       WHEN CAST(total_comparison AS DOUBLE) / CAST(total_2019 AS DOUBLE) * 100.0 < 100 THEN 'below_2019'
       WHEN CAST(total_comparison AS DOUBLE) / CAST(total_2019 AS DOUBLE) * 100.0 = 100 THEN 'unchanged'
       ELSE 'above_2019' END AS overall_recovery_class,
  -- Domestic
  domestic_travelers_2019, domestic_travelers_comparison,
  CASE WHEN domestic_travelers_2019 IS NOT NULL AND domestic_travelers_2019 > 0 AND domestic_travelers_comparison IS NOT NULL AND domestic_travelers_comparison >= 0 AND comparable_to_2019 IS TRUE
    THEN CAST(domestic_travelers_comparison AS DOUBLE) / CAST(domestic_travelers_2019 AS DOUBLE) * 100.0 ELSE NULL END AS domestic_recovery_index,
  CASE WHEN domestic_travelers_2019 IS NULL OR domestic_travelers_2019 <= 0 OR domestic_travelers_comparison IS NULL OR domestic_travelers_comparison < 0 OR comparable_to_2019 IS NOT TRUE THEN NULL
       WHEN CAST(domestic_travelers_comparison AS DOUBLE) / CAST(domestic_travelers_2019 AS DOUBLE) * 100.0 < 100 THEN 'below_2019'
       WHEN CAST(domestic_travelers_comparison AS DOUBLE) / CAST(domestic_travelers_2019 AS DOUBLE) * 100.0 = 100 THEN 'unchanged'
       ELSE 'above_2019' END AS domestic_recovery_class,
  CASE WHEN domestic_travelers_2019 IS NOT NULL AND domestic_travelers_2019 > 0 AND domestic_travelers_comparison IS NOT NULL AND domestic_travelers_comparison >= 0 AND comparable_to_2019 IS TRUE THEN TRUE ELSE FALSE END AS domestic_eligible,
  CASE WHEN domestic_travelers_2019 IS NULL THEN 'domestic baseline is NULL'
       WHEN domestic_travelers_2019 <= 0 THEN 'domestic baseline is nonpositive'
       WHEN domestic_travelers_comparison IS NULL THEN 'domestic comparison is NULL'
       WHEN domestic_travelers_comparison < 0 THEN 'domestic comparison is negative'
       WHEN comparable_to_2019 IS NOT TRUE THEN 'not comparable to 2019'
       ELSE NULL END AS domestic_exclusion_reason,
  -- Foreign (BLOCKED: foreign_includes_overseas_filipinos not available in Gold; non-NULL foreign values do not prove separate category comparability)
  foreign_travelers_2019, foreign_travelers_comparison,
  CAST(NULL AS DOUBLE) AS foreign_recovery_index,
  CAST(NULL AS STRING) AS foreign_recovery_class,
  FALSE AS foreign_eligible,
  'foreign_includes_overseas_filipinos not available in Gold; separate foreign segment analysis blocked' AS foreign_exclusion_reason,
  -- Overseas Filipino (BLOCKED: foreign_includes_overseas_filipinos not available in Gold; separate overseas Filipino counts not independently comparable)
  overseas_filipinos_2019, overseas_filipinos_comparison,
  CAST(NULL AS DOUBLE) AS overseas_filipino_recovery_index,
  CAST(NULL AS STRING) AS overseas_filipino_recovery_class,
  FALSE AS overseas_filipino_eligible,
  'foreign_includes_overseas_filipinos not available in Gold; separate overseas Filipino segment analysis blocked' AS overseas_filipino_exclusion_reason,
  -- Category compatibility tracking
  FALSE AS foreign_segment_supported,
  FALSE AS overseas_filipino_segment_supported,
  'foreign_includes_overseas_filipinos missing from Gold (DL-019); separate foreign/overseas Filipino analysis blocked' AS category_block_reason,
  -- Below-2019 flags
  CASE WHEN total_2019 IS NOT NULL AND total_2019 > 0 AND total_comparison IS NOT NULL AND total_comparison >= 0 AND comparable_to_2019 IS TRUE
        AND CAST(total_comparison AS DOUBLE) / CAST(total_2019 AS DOUBLE) * 100.0 < 100 THEN TRUE ELSE FALSE END AS below_2019_total_flag,
  CASE WHEN total_2019 IS NOT NULL AND total_2019 > 0 AND total_comparison IS NOT NULL AND total_comparison >= 0 AND comparable_to_2019 IS TRUE
        AND CAST(total_comparison AS DOUBLE) / CAST(total_2019 AS DOUBLE) * 100.0 < 100 THEN TRUE ELSE FALSE END AS segment_analysis_cohort_flag,
  -- Caveats
  baseline_traveler_value_status, comparison_traveler_value_status, comparable_to_2019, comparability_note
FROM joined
ORDER BY province_name, comparison_year

### F2: Validate `traveler_segment_recovery`

This cell runs six blocking checks:

| Check | What it validates | Failure consequence |
| --- | --- | --- |
| F1 | Natural key (`province_psgc`, `comparison_year`) is unique and non-NULL | BLOCK: grain is wrong |
| F2 | Expected 200 rows (100 units × 2 years) | BLOCK: missing or extra rows |
| F3 | No segment recovery with invalid baseline (segment baseline ≤ 0 or NULL when segment_eligible) | BLOCK: eligibility logic wrong |
| F4 | No segment recovery when `comparable_to_2019` is not TRUE | BLOCK: comparability not checked |
| F5 | Ineligible segments have NULL recovery metrics | BLOCK: metrics leaked to ineligible |
| F6 | `below_2019_total_flag` is TRUE only when overall recovery is eligible and below 2019 | BLOCK: flag logic wrong |

**Input:** Temp view `v_traveler_segment_recovery`.

**Output:** Temp view `v_val_f_segment`.

**Failure consequence:** BLOCK: the `traveler_segment_recovery` table is not replaced.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_f_segment AS
WITH checks AS (
  SELECT 'F1_dup_keys' AS check_id, COUNT(*) AS failures FROM (
    SELECT province_psgc, comparison_year FROM v_traveler_segment_recovery GROUP BY province_psgc, comparison_year HAVING COUNT(*) > 1
  )
  UNION ALL
  SELECT 'F1_null_keys', COUNT(*) FROM v_traveler_segment_recovery WHERE province_psgc IS NULL OR comparison_year IS NULL
  UNION ALL
  SELECT 'F2_row_count', CASE WHEN COUNT(*) != 200 THEN 1 ELSE 0 END FROM v_traveler_segment_recovery
  UNION ALL
  SELECT 'F3_bad_domestic_baseline', COUNT(*) FROM v_traveler_segment_recovery
  WHERE domestic_eligible AND (domestic_travelers_2019 IS NULL OR domestic_travelers_2019 <= 0)
  UNION ALL
  SELECT 'F3_bad_foreign_baseline', COUNT(*) FROM v_traveler_segment_recovery
  WHERE foreign_eligible AND (foreign_travelers_2019 IS NULL OR foreign_travelers_2019 <= 0)
  UNION ALL
  SELECT 'F3_bad_of_baseline', COUNT(*) FROM v_traveler_segment_recovery
  WHERE overseas_filipino_eligible AND (overseas_filipinos_2019 IS NULL OR overseas_filipinos_2019 <= 0)
  UNION ALL
  SELECT 'F4_bad_comparable_dom', COUNT(*) FROM v_traveler_segment_recovery
  WHERE domestic_eligible AND comparable_to_2019 IS NOT TRUE
  UNION ALL
  SELECT 'F5_metrics_ineligible_dom', COUNT(*) FROM v_traveler_segment_recovery
  WHERE NOT domestic_eligible AND (domestic_recovery_index IS NOT NULL OR domestic_recovery_class IS NOT NULL)
  UNION ALL
  SELECT 'F5_metrics_ineligible_for', COUNT(*) FROM v_traveler_segment_recovery
  WHERE NOT foreign_eligible AND (foreign_recovery_index IS NOT NULL OR foreign_recovery_class IS NOT NULL)
  UNION ALL
  SELECT 'F5_metrics_ineligible_of', COUNT(*) FROM v_traveler_segment_recovery
  WHERE NOT overseas_filipino_eligible AND (overseas_filipino_recovery_index IS NOT NULL OR overseas_filipino_recovery_class IS NOT NULL)
  UNION ALL
  SELECT 'F6_bad_below_flag', COUNT(*) FROM v_traveler_segment_recovery
  WHERE below_2019_total_flag AND (overall_recovery_class IS NULL OR overall_recovery_class != 'below_2019')
)
SELECT * FROM checks

### F3: Assert `traveler_segment_recovery` checks and persist

This cell collects the check results from F2. If any check has failures > 0, it raises an error and the table is **not** replaced.

**Input:** Temp view `v_val_f_segment`.

**Output:** `tuklas_dev`.`05_analytics`.`traveler_segment_recovery` is replaced (if all checks pass).

In [0]:
%python
for row in spark.table("v_val_f_segment").collect():
    val_check(row.check_id, "v_traveler_segment_recovery", row.failures,
              f"{row.check_id}: {row.failures} failures", phase="Phase F")
assert_blocking("Phase F -- traveler_segment_recovery validation")
record_output("traveler_segment_recovery", spark.table("v_traveler_segment_recovery").count())

### F4: Persist `traveler_segment_recovery`

This cell replaces the persisted Delta table `tuklas_dev`.`05_analytics`.`traveler_segment_recovery` with the validated candidate view.

**Input:** Temp view `v_traveler_segment_recovery` (validated by F2–F3).

**Output:** Persisted Delta table with grain (province_psgc, comparison_year) and 200 rows.

**Rerun:** Idempotent: replaces the table on each run.

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`
USING DELTA
AS SELECT
  province_psgc, province_name, unit_type, region_psgc, region_name, comparison_year,
  overall_recovery_index, overall_recovery_class,
  domestic_travelers_2019, domestic_travelers_comparison,
  domestic_recovery_index, domestic_recovery_class, domestic_eligible, domestic_exclusion_reason,
  foreign_travelers_2019, foreign_travelers_comparison,
  foreign_recovery_index, foreign_recovery_class, foreign_eligible, foreign_exclusion_reason,
  overseas_filipinos_2019, overseas_filipinos_comparison,
  overseas_filipino_recovery_index, overseas_filipino_recovery_class, overseas_filipino_eligible, overseas_filipino_exclusion_reason,
  foreign_segment_supported, overseas_filipino_segment_supported, category_block_reason,
  below_2019_total_flag, segment_analysis_cohort_flag,
  baseline_traveler_value_status, comparison_traveler_value_status, comparable_to_2019, comparability_note
FROM v_traveler_segment_recovery

## Phase G: Tourism and economic alignment

**Purpose:** Build `tourism_economy` that answers BQ2 -- *How is recovery in tourism activity associated with changes in local tourism-related economic activity across provinces?*

**Supported BQ/AQ:**
- BQ2-AQ2.1: Traveler change versus real A&F GVA growth
- BQ2-AQ2.2: Matching versus divergent directions
- BQ2-AQ2.3: A&F economic significance across recovery patterns
Additional context: 2023 tourism intensity versus 2023 GDP per capita.

**Grain:** One reporting unit for the 2019–2023 comparison (100 rows).

**Formulas:**
- `tourism_change_pct` = (`travelers_2023` − `travelers_2019`) / `travelers_2019` × 100
- `afs_gva_growth_pct` = (`afs_gva_constant_2023` − `afs_gva_constant_2019`) / `afs_gva_constant_2019` × 100 (CONSTANT prices for growth)
- `afs_share_of_gdp_pct` = `afs_gva_current_2023` / `gdp_total_current_2023` × 100 (CURRENT prices for share)
- `tourism_intensity_per_1000_2023` = `travelers_2023` / `population_total_2023` × 1000

**Eligibility (defined independently):**
- Tourism change: `travelers_2019` > 0, `travelers_2023` is not NULL and nonnegative, and `comparable_to_2019` IS TRUE. FALSE or NULL comparability makes the comparison ineligible.
- A&F growth: `afs_gva_constant_2019` > 0 and `afs_gva_constant_2023` is not NULL.
- A&F share: `afs_gva_current_2023` is not NULL and `gdp_total_current_2023` > 0.
- Intensity/GDP context: `travelers_2023` is not NULL, `population_total_2023` > 0, `gdp_per_capita_current_2023` is not NULL. Same eligibility as `tourism_activity` (non-NULL status, not `not_reported`).

**Key rule:** Missing tourism comparability must not suppress a valid 2023 A&F share. Missing A&F growth must not suppress valid tourism recovery. Each metric's eligibility is independent.

**Alignment eligibility:** Requires BOTH valid tourism change AND valid A&F growth. `alignment_eligible` and `alignment_exclusion_reason` are provided.

**Comparability enforcement:** When `comparable_to_2019` IS NOT TRUE, `recovery_index_2023`, `recovery_class_2023`, `tourism_change_pct`, `tourism_direction`, and `tourism_economy_pattern` are set to NULL. A blocking check verifies that non-comparable units (e.g., Aklan) cannot receive valid tourism recovery or alignment results.

**Direction:**
- Positive: `increased`
- Zero: `unchanged`
- Negative: `decreased`
- Ineligible: NULL

**Patterns (machine-readable codes):**
- `tourism_up_economy_up`, `tourism_up_economy_down`, `tourism_down_economy_up`, `tourism_down_economy_down`
- `unchanged` if either eligible direction is `unchanged`
- NULL if alignment is ineligible (either direction is NULL)

**Pattern labels (`tourism_economy_pattern_label`):**
- `tourism_up_economy_up` — "Tourism activity increased; real A&F GVA increased"
- `tourism_up_economy_down` — "Tourism activity increased; real A&F GVA decreased"
- `tourism_down_economy_up` — "Tourism activity decreased; real A&F GVA increased"
- `tourism_down_economy_down` — "Tourism activity decreased; real A&F GVA decreased"
- When either direction is `unchanged`, both directions are described explicitly (e.g., "Tourism activity unchanged; real A&F GVA increased")
- NULL if alignment is ineligible (either direction is NULL)

**Definition:** Tourism activity means reported overnight travelers, as defined by DOT.

**Note:** The machine-readable pattern code (`tourism_economy_pattern`) is unchanged. The presentation label (`tourism_economy_pattern_label`) is a separate column for human-readable output.

**Context column:** `gdp_per_capita_current_2019` is exposed for baseline economic context. No high/low or priority thresholds are invented; this is proposed context, not an approved AQ.

**Caveats preserved from Gold:** `baseline_traveler_value_status`, `comparison_traveler_value_status`, `comparable_to_2019`, `population_total_2023`, `population_source_2023`. Missing fields (`derived_values` from DL-024, `per_capita_recomputed` from DL-011) are reported, not invented.

**Input:** `fact_province_year` (2019, 2023) and `dim_province`.

### G1: Build `tourism_economy` candidate

This cell creates a temporary view `v_tourism_economy` with one row per reporting unit (100 rows). It:
- Joins 2019 baseline to 2023 comparison data on `province_psgc`.
- Carries `baseline_traveler_value_status`, `comparison_traveler_value_status`, `comparable_to_2019`, `population_total_2023`, `population_source_2023` from Gold.
- Computes `recovery_index_2023` and `recovery_class_2023` using the same rules as `tourism_recovery` (requires `comparable_to_2019` IS TRUE).
- Computes `tourism_change_pct`, `afs_gva_growth_pct`, `afs_share_of_gdp_pct`, `tourism_intensity_per_1000_2023`.
- Uses the same 2023 intensity eligibility and formula as `tourism_activity` (non-NULL status, not `not_reported`, population > 0).
- Determines `tourism_direction` and `economy_direction` independently.
- Classifies `tourism_economy_pattern` from the two directions (NULL if either is NULL).
- Adds `tourism_economy_pattern_label` with human-readable wording: "Tourism activity <direction>; real A&F GVA <direction>" (NULL if ineligible).
- Tourism activity means reported overnight travelers, as defined by DOT.
- Adds `alignment_eligible` and `alignment_exclusion_reason` (requires both valid tourism change and valid A&F growth).
- When `comparable_to_2019` IS NOT TRUE, tourism metrics are NULL and alignment is ineligible.
- Preserves eligibility flags, exclusion reasons, and `comparability_note`.

**Input:** `fact_province_year` (2019, 2023) and `dim_province`.

**Output:** Temp view `v_tourism_economy`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_tourism_economy AS
WITH baseline AS (
  SELECT province_psgc,
    total_travelers AS travelers_2019,
    afs_gva_constant AS afs_gva_constant_2019,
    gdp_per_capita_current AS gdp_per_capita_current_2019,
    traveler_value_status AS baseline_traveler_value_status
  FROM `tuklas_dev`.`04_gold`.`fact_province_year` WHERE year = 2019
),
comparison AS (
  SELECT province_psgc,
    total_travelers AS travelers_2023,
    afs_gva_constant AS afs_gva_constant_2023,
    afs_gva_current AS afs_gva_current_2023,
    gdp_total_current AS gdp_total_current_2023,
    gdp_per_capita_current AS gdp_per_capita_current_2023,
    population_total AS population_total_2023,
    population_source AS population_source_2023,
    traveler_value_status AS comparison_traveler_value_status,
    comparable_to_2019 AS comparison_comparable_to_2019
  FROM `tuklas_dev`.`04_gold`.`fact_province_year` WHERE year = 2023
),
joined AS (
  SELECT
    p.province_psgc, p.province_name, p.unit_type, p.region_psgc, p.region_name, p.comparability_note,
    b.travelers_2019, c.travelers_2023,
    b.afs_gva_constant_2019, c.afs_gva_constant_2023,
    c.afs_gva_current_2023, c.gdp_total_current_2023,
    b.gdp_per_capita_current_2019, c.gdp_per_capita_current_2023,
    c.population_total_2023, c.population_source_2023,
    b.baseline_traveler_value_status, c.comparison_traveler_value_status,
    c.comparison_comparable_to_2019 AS comparable_to_2019
  FROM baseline b
  JOIN comparison c ON b.province_psgc = c.province_psgc
  JOIN `tuklas_dev`.`04_gold`.`dim_province` p ON b.province_psgc = p.province_psgc
),
calc AS (
SELECT
  province_psgc, province_name, unit_type, region_psgc, region_name,
  2019 AS baseline_year, 2023 AS comparison_year,
  travelers_2019, travelers_2023,
  -- Recovery index 2023 (requires comparable_to_2019 IS TRUE, same rules as tourism_recovery)
  CASE WHEN travelers_2019 IS NOT NULL AND travelers_2019 > 0 AND travelers_2023 IS NOT NULL AND travelers_2023 >= 0 AND comparable_to_2019 IS TRUE
    THEN CAST(travelers_2023 AS DOUBLE) / CAST(travelers_2019 AS DOUBLE) * 100.0 ELSE NULL END AS recovery_index_2023,
  CASE WHEN travelers_2019 IS NULL OR travelers_2019 <= 0 OR travelers_2023 IS NULL OR travelers_2023 < 0 OR comparable_to_2019 IS NOT TRUE THEN NULL
       WHEN CAST(travelers_2023 AS DOUBLE) / CAST(travelers_2019 AS DOUBLE) * 100.0 < 100 THEN 'below_2019'
       WHEN CAST(travelers_2023 AS DOUBLE) / CAST(travelers_2019 AS DOUBLE) * 100.0 = 100 THEN 'unchanged'
       ELSE 'above_2019' END AS recovery_class_2023,
  -- Tourism change pct (requires comparable_to_2019 IS TRUE)
  CASE WHEN travelers_2019 IS NOT NULL AND travelers_2019 > 0 AND travelers_2023 IS NOT NULL AND comparable_to_2019 IS TRUE
    THEN (CAST(travelers_2023 AS DOUBLE) - CAST(travelers_2019 AS DOUBLE)) / CAST(travelers_2019 AS DOUBLE) * 100.0
    ELSE NULL END AS tourism_change_pct,
  afs_gva_constant_2019, afs_gva_constant_2023,
  -- A&F growth pct (constant prices)
  CASE WHEN afs_gva_constant_2019 IS NOT NULL AND afs_gva_constant_2019 > 0 AND afs_gva_constant_2023 IS NOT NULL
    THEN (CAST(afs_gva_constant_2023 AS DOUBLE) - CAST(afs_gva_constant_2019 AS DOUBLE)) / CAST(afs_gva_constant_2019 AS DOUBLE) * 100.0
    ELSE NULL END AS afs_gva_growth_pct,
  -- Tourism direction (requires comparable_to_2019 IS TRUE)
  CASE WHEN travelers_2019 IS NULL OR travelers_2019 <= 0 OR travelers_2023 IS NULL OR comparable_to_2019 IS NOT TRUE THEN NULL
       WHEN travelers_2023 > travelers_2019 THEN 'increased'
       WHEN travelers_2023 = travelers_2019 THEN 'unchanged'
       ELSE 'decreased' END AS tourism_direction,
  -- Economy direction
  CASE WHEN afs_gva_constant_2019 IS NULL OR afs_gva_constant_2019 <= 0 OR afs_gva_constant_2023 IS NULL THEN NULL
       WHEN afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'increased'
       WHEN afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'unchanged'
       ELSE 'decreased' END AS economy_direction,
  -- A&F current values
  afs_gva_current_2023, gdp_total_current_2023,
  -- A&F share of GDP (current prices)
  CASE WHEN afs_gva_current_2023 IS NOT NULL AND gdp_total_current_2023 IS NOT NULL AND gdp_total_current_2023 > 0
    THEN CAST(afs_gva_current_2023 AS DOUBLE) / CAST(gdp_total_current_2023 AS DOUBLE) * 100.0
    ELSE NULL END AS afs_share_of_gdp_pct,
  -- GDP per capita
  gdp_per_capita_current_2023, gdp_per_capita_current_2019,
  -- Tourism intensity 2023 (same eligibility as tourism_activity: non-NULL status, not not_reported, population > 0)
  CASE WHEN travelers_2023 IS NOT NULL AND travelers_2023 >= 0
           AND comparison_traveler_value_status IS NOT NULL AND comparison_traveler_value_status != 'not_reported'
           AND population_total_2023 IS NOT NULL AND population_total_2023 > 0
    THEN CAST(travelers_2023 AS DOUBLE) / CAST(population_total_2023 AS DOUBLE) * 1000.0
    ELSE NULL END AS tourism_intensity_per_1000_2023,
  -- Intensity eligibility 2023 (same as tourism_activity)
  CASE WHEN travelers_2023 IS NULL THEN FALSE
       WHEN travelers_2023 < 0 THEN FALSE
       WHEN comparison_traveler_value_status IS NULL THEN FALSE
       WHEN comparison_traveler_value_status = 'not_reported' THEN FALSE
       WHEN population_total_2023 IS NULL THEN FALSE
       WHEN population_total_2023 <= 0 THEN FALSE
       ELSE TRUE END AS intensity_eligible_2023,
  CASE WHEN travelers_2023 IS NULL THEN 'travelers_2023 is NULL'
       WHEN travelers_2023 < 0 THEN 'travelers_2023 is negative'
       WHEN comparison_traveler_value_status IS NULL THEN 'comparison_traveler_value_status is NULL'
       WHEN comparison_traveler_value_status = 'not_reported' THEN 'comparison_traveler_value_status is not_reported'
       WHEN population_total_2023 IS NULL THEN 'population_total_2023 is NULL'
       WHEN population_total_2023 <= 0 THEN 'population_total_2023 is nonpositive'
       ELSE NULL END AS intensity_exclusion_reason_2023,
  -- Eligibility flags (tourism_change requires comparable_to_2019 IS TRUE, same as tourism_recovery)
  CASE WHEN travelers_2019 IS NOT NULL AND travelers_2019 > 0 AND travelers_2023 IS NOT NULL AND travelers_2023 >= 0 AND comparable_to_2019 IS TRUE THEN TRUE ELSE FALSE END AS tourism_change_eligible,
  CASE WHEN travelers_2019 IS NULL THEN 'travelers_2019 is NULL'
       WHEN travelers_2019 <= 0 THEN 'travelers_2019 is nonpositive'
       WHEN travelers_2023 IS NULL THEN 'travelers_2023 is NULL'
       WHEN travelers_2023 < 0 THEN 'travelers_2023 is negative'
       WHEN comparable_to_2019 IS NOT TRUE THEN 'comparison year not comparable to 2019'
       ELSE NULL END AS tourism_change_exclusion_reason,
  CASE WHEN afs_gva_constant_2019 IS NOT NULL AND afs_gva_constant_2019 > 0 AND afs_gva_constant_2023 IS NOT NULL THEN TRUE ELSE FALSE END AS afs_growth_eligible,
  CASE WHEN afs_gva_constant_2019 IS NULL THEN 'afs_gva_constant_2019 is NULL'
       WHEN afs_gva_constant_2019 <= 0 THEN 'afs_gva_constant_2019 is nonpositive'
       WHEN afs_gva_constant_2023 IS NULL THEN 'afs_gva_constant_2023 is NULL' ELSE NULL END AS afs_growth_exclusion_reason,
  CASE WHEN afs_gva_current_2023 IS NOT NULL AND gdp_total_current_2023 IS NOT NULL AND gdp_total_current_2023 > 0 THEN TRUE ELSE FALSE END AS afs_share_eligible,
  CASE WHEN afs_gva_current_2023 IS NULL THEN 'afs_gva_current_2023 is NULL'
       WHEN gdp_total_current_2023 IS NULL THEN 'gdp_total_current_2023 is NULL'
       WHEN gdp_total_current_2023 <= 0 THEN 'gdp_total_current_2023 is nonpositive' ELSE NULL END AS afs_share_exclusion_reason,
  CASE WHEN travelers_2023 IS NOT NULL AND population_total_2023 IS NOT NULL AND population_total_2023 > 0 AND gdp_per_capita_current_2023 IS NOT NULL THEN TRUE ELSE FALSE END AS intensity_gdp_context_eligible,
  CASE WHEN travelers_2023 IS NULL THEN 'travelers_2023 is NULL'
       WHEN population_total_2023 IS NULL THEN 'population_total_2023 is NULL'
       WHEN population_total_2023 <= 0 THEN 'population_total_2023 is nonpositive'
       WHEN gdp_per_capita_current_2023 IS NULL THEN 'gdp_per_capita_current_2023 is NULL' ELSE NULL END AS intensity_gdp_context_exclusion_reason,
  comparability_note,
  baseline_traveler_value_status, comparison_traveler_value_status, comparable_to_2019,
  population_total_2023, population_source_2023
FROM joined
)
SELECT calc.*,
  CASE WHEN calc.tourism_direction IS NULL OR calc.economy_direction IS NULL THEN NULL
       WHEN calc.tourism_direction = 'unchanged' OR calc.economy_direction = 'unchanged' THEN 'unchanged'
       WHEN calc.tourism_direction = 'increased' AND calc.economy_direction = 'increased' THEN 'tourism_up_economy_up'
       WHEN calc.tourism_direction = 'increased' AND calc.economy_direction = 'decreased' THEN 'tourism_up_economy_down'
       WHEN calc.tourism_direction = 'decreased' AND calc.economy_direction = 'increased' THEN 'tourism_down_economy_up'
       WHEN calc.tourism_direction = 'decreased' AND calc.economy_direction = 'decreased' THEN 'tourism_down_economy_down'
  END AS tourism_economy_pattern,
  CASE WHEN calc.tourism_direction IS NULL OR calc.economy_direction IS NULL THEN NULL
       WHEN calc.tourism_direction = 'increased' AND calc.economy_direction = 'increased' THEN 'Tourism activity increased; real A&F GVA increased'
       WHEN calc.tourism_direction = 'increased' AND calc.economy_direction = 'decreased' THEN 'Tourism activity increased; real A&F GVA decreased'
       WHEN calc.tourism_direction = 'increased' AND calc.economy_direction = 'unchanged' THEN 'Tourism activity increased; real A&F GVA unchanged'
       WHEN calc.tourism_direction = 'decreased' AND calc.economy_direction = 'increased' THEN 'Tourism activity decreased; real A&F GVA increased'
       WHEN calc.tourism_direction = 'decreased' AND calc.economy_direction = 'decreased' THEN 'Tourism activity decreased; real A&F GVA decreased'
       WHEN calc.tourism_direction = 'decreased' AND calc.economy_direction = 'unchanged' THEN 'Tourism activity decreased; real A&F GVA unchanged'
       WHEN calc.tourism_direction = 'unchanged' AND calc.economy_direction = 'increased' THEN 'Tourism activity unchanged; real A&F GVA increased'
       WHEN calc.tourism_direction = 'unchanged' AND calc.economy_direction = 'decreased' THEN 'Tourism activity unchanged; real A&F GVA decreased'
       WHEN calc.tourism_direction = 'unchanged' AND calc.economy_direction = 'unchanged' THEN 'Tourism activity unchanged; real A&F GVA unchanged'
  END AS tourism_economy_pattern_label,
  CASE WHEN calc.tourism_change_eligible AND calc.afs_growth_eligible THEN TRUE ELSE FALSE END AS alignment_eligible,
  CASE WHEN NOT calc.tourism_change_eligible AND NOT calc.afs_growth_eligible THEN 'neither tourism change nor A&F growth eligible'
       WHEN NOT calc.tourism_change_eligible THEN 'tourism change not eligible'
       WHEN NOT calc.afs_growth_eligible THEN 'A&F growth not eligible'
       ELSE NULL END AS alignment_exclusion_reason
FROM calc
ORDER BY province_name

### G2: Validate `tourism_economy`

This cell runs twelve checks:

| Check | What it validates | Severity | Failure consequence |
| --- | --- | --- | --- |
| G1 | Natural key `province_psgc` is unique and non-NULL | BLOCK |: grain is wrong |
| G2 | Expected 100 rows | BLOCK |: missing or extra rows |
| G3 | No `tourism_economy_pattern` when either direction is NULL | BLOCK |: pattern computed despite ineligible inputs |
| G4 | `afs_share_of_gdp_pct` outside 0–100% | REVIEW |: flagged for review without altering values |
| G5 | `afs_share_eligible` is TRUE even when `afs_growth_eligible` is FALSE (if inputs are present) | BLOCK |: missing growth suppressed valid share |
| G6 | `recovery_class_2023` is one of `below_2019`, `unchanged`, `above_2019`, or NULL | BLOCK |: invalid classification |
| G7 | Non-comparable units have no recovery metrics or tourism change | BLOCK |: non-comparable unit received valid recovery |
| G8 | Non-comparable units are not alignment-eligible | BLOCK |: non-comparable unit received alignment |
| G9 | Recovery index/class match `tourism_recovery` for 2023 | BLOCK |: cross-table mismatch |
| G10 | Intensity and eligibility match `tourism_activity` for 2023 | BLOCK |: cross-table mismatch |
| G11 | Base values (travelers, A&F GVA, population) match Gold | BLOCK |: base values diverged from Gold |
| G12 | `tourism_economy_pattern_label` is NULL iff `tourism_economy_pattern` is NULL | BLOCK |: label inconsistent with pattern eligibility |

**Input:** Temp view `v_tourism_economy`.

**Output:** Temp view `v_val_g_economy`.

**Failure consequence:** BLOCK checks stop execution; REVIEW check G4 reports the issue but does not stop.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_g_economy AS
WITH checks AS (
  SELECT 'G1_dup_keys' AS check_id, 'BLOCK' AS severity, COUNT(*) AS failures FROM (
    SELECT province_psgc FROM v_tourism_economy GROUP BY province_psgc HAVING COUNT(*) > 1
  )
  UNION ALL
  SELECT 'G1_null_keys', 'BLOCK', COUNT(*) FROM v_tourism_economy WHERE province_psgc IS NULL
  UNION ALL
  SELECT 'G2_row_count', 'BLOCK', CASE WHEN COUNT(*) != 100 THEN 1 ELSE 0 END FROM v_tourism_economy
  UNION ALL
  SELECT 'G3_pattern_ineligible', 'BLOCK', COUNT(*) FROM v_tourism_economy
  WHERE tourism_economy_pattern IS NOT NULL AND (tourism_direction IS NULL OR economy_direction IS NULL)
  UNION ALL
  SELECT 'G4_share_outside_range', 'REVIEW', COUNT(*) FROM v_tourism_economy
  WHERE afs_share_of_gdp_pct IS NOT NULL AND (afs_share_of_gdp_pct < 0 OR afs_share_of_gdp_pct > 100)
  UNION ALL
  SELECT 'G5_share_suppressed', 'BLOCK', COUNT(*) FROM v_tourism_economy
  WHERE afs_gva_current_2023 IS NOT NULL AND gdp_total_current_2023 IS NOT NULL AND gdp_total_current_2023 > 0
    AND afs_share_eligible IS NOT TRUE
  UNION ALL
  SELECT 'G6_bad_class', 'BLOCK', COUNT(*) FROM v_tourism_economy
  WHERE recovery_class_2023 IS NOT NULL
    AND recovery_class_2023 NOT IN ('below_2019', 'unchanged', 'above_2019')
  UNION ALL
  SELECT 'G7_noncomparable_has_recovery', 'BLOCK', COUNT(*) FROM v_tourism_economy
  WHERE comparable_to_2019 IS NOT TRUE AND (recovery_index_2023 IS NOT NULL OR recovery_class_2023 IS NOT NULL OR tourism_change_pct IS NOT NULL OR tourism_direction IS NOT NULL)
  UNION ALL
  SELECT 'G8_noncomparable_aligned', 'BLOCK', COUNT(*) FROM v_tourism_economy
  WHERE comparable_to_2019 IS NOT TRUE AND alignment_eligible IS TRUE
  UNION ALL
  SELECT 'G9_recovery_vs_tourism_recovery', 'BLOCK', COUNT(*) FROM (
    SELECT e.province_psgc
    FROM v_tourism_economy e
    JOIN v_tourism_recovery r ON e.province_psgc = r.province_psgc AND r.comparison_year = 2023
    WHERE NOT (e.recovery_index_2023 <=> r.recovery_index)
       OR NOT (e.recovery_class_2023 <=> r.recovery_class)
  )
  UNION ALL
  SELECT 'G10_intensity_vs_activity', 'BLOCK', COUNT(*) FROM (
    SELECT e.province_psgc
    FROM v_tourism_economy e
    JOIN v_tourism_activity a ON e.province_psgc = a.province_psgc AND a.year = 2023
    WHERE NOT (e.tourism_intensity_per_1000_2023 <=> a.tourism_intensity_per_1000)
       OR NOT (e.intensity_eligible_2023 <=> a.intensity_eligible)
  )
  UNION ALL
  SELECT 'G11_base_vs_gold', 'BLOCK', COUNT(*) FROM (
    SELECT e.province_psgc
    FROM v_tourism_economy e
    JOIN `tuklas_dev`.`04_gold`.`fact_province_year` b ON e.province_psgc = b.province_psgc AND b.year = 2019
    JOIN `tuklas_dev`.`04_gold`.`fact_province_year` c ON e.province_psgc = c.province_psgc AND c.year = 2023
    WHERE NOT (e.travelers_2019 <=> b.total_travelers)
       OR NOT (e.travelers_2023 <=> c.total_travelers)
       OR NOT (e.afs_gva_constant_2019 <=> b.afs_gva_constant)
       OR NOT (e.afs_gva_constant_2023 <=> c.afs_gva_constant)
       OR NOT (e.population_total_2023 <=> c.population_total)
  )
  UNION ALL
  SELECT 'G12_label_pattern_consistency', 'BLOCK', COUNT(*) FROM v_tourism_economy
  WHERE (tourism_economy_pattern IS NULL AND tourism_economy_pattern_label IS NOT NULL)
     OR (tourism_economy_pattern IS NOT NULL AND tourism_economy_pattern_label IS NULL)
)
SELECT * FROM checks

### G3: Assert `tourism_economy` checks and persist

This cell collects the check results from G2. BLOCK checks raise an error if they fail. REVIEW check G4 is reported but does not stop execution: the A&F share values are not altered.

**Input:** Temp view `v_val_g_economy`.

**Output:** `tuklas_dev`.`05_analytics`.`tourism_economy` is replaced (if all BLOCK checks pass).

In [0]:
%python
blocks = []
for row in spark.table("v_val_g_economy").collect():
    is_block = (row.severity == 'BLOCK')
    val_check(row.check_id, "v_tourism_economy", row.failures,
              f"{row.check_id} ({row.severity}): {row.failures} failures" if row.failures else f"{row.check_id} ({row.severity}): PASS",
              severity=row.severity, phase="Phase G")
    if is_block and row.failures > 0:
        blocks.append((row.check_id, row.failures))

if blocks:
    msg = "; ".join(f"{b[0]}: {b[1]} failures" for b in blocks)
    raise RuntimeError(f"BLOCKING checks failed in Phase G -- tourism_economy: {msg}")
print("  -> All Phase G BLOCK checks passed. REVIEW flags reported above.")
record_output("tourism_economy", spark.table("v_tourism_economy").count())

### G4: Persist `tourism_economy`

This cell replaces the persisted Delta table `tuklas_dev`.`05_analytics`.`tourism_economy` with the validated candidate view.

**Input:** Temp view `v_tourism_economy` (validated by G2–G3).

**Output:** Persisted Delta table with grain (province_psgc) and 100 rows.

**Rerun:** Idempotent: replaces the table on each run.

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`tourism_economy`
USING DELTA
AS SELECT
  province_psgc, province_name, unit_type, region_psgc, region_name,
  baseline_year, comparison_year,
  travelers_2019, travelers_2023,
  recovery_index_2023, recovery_class_2023,
  tourism_change_pct,
  afs_gva_constant_2019, afs_gva_constant_2023, afs_gva_growth_pct,
  tourism_direction, economy_direction, tourism_economy_pattern, tourism_economy_pattern_label,
  afs_gva_current_2023, gdp_total_current_2023, afs_share_of_gdp_pct,
  gdp_per_capita_current_2023, gdp_per_capita_current_2019,
  tourism_intensity_per_1000_2023,
  intensity_eligible_2023, intensity_exclusion_reason_2023,
  alignment_eligible, alignment_exclusion_reason,
  baseline_traveler_value_status, comparison_traveler_value_status, comparable_to_2019,
  population_total_2023, population_source_2023,
  tourism_change_eligible, tourism_change_exclusion_reason,
  afs_growth_eligible, afs_growth_exclusion_reason,
  afs_share_eligible, afs_share_exclusion_reason,
  intensity_gdp_context_eligible, intensity_gdp_context_exclusion_reason,
  comparability_note
FROM v_tourism_economy

## Phase H: Final validation and manual spot checks

**Purpose:** Run cross-table validation checks that verify Analytics base values match Gold, and provide manual spot-check queries for specific provinces and edge cases.

**Supported BQ/AQ:** All (guards data integrity across all five tables).

**Cross-table checks:**
- H1: `tourism_activity` base values match Gold `fact_province_year` using null-safe comparison (`\<=>`).
- H2: `tourism_recovery` base values (`travelers_2019`, `travelers_comparison_year`, `comparable_to_2019`) match Gold.
- H3: `tourism_economy` base values match Gold for both 2019 and 2023.
- H4: `traveler_segment_recovery` `overall_recovery_index` matches `tourism_recovery.recovery_index`.

**Manual spot checks** for:
- Quezon (province with reported values)
- Aklan (comparability FALSE from 2021: OI-4 Boracay)
- Pampanga (province with `dash_as_zero` status)
- City of Angeles (separately reported HUC)
- Basilan (unit with missing DOT values in 2019: `not_reported`)
- Aklan 2022 (unit/year with `comparable_to_2019` = FALSE)

### H1: Cross-table validation checks

This cell creates a temp view `v_val_h_cross` with four checks using null-safe comparison (`\<=>`). Null-safe comparison treats NULL = NULL as TRUE, NULL vs a value as FALSE, and does not convert NULL to 0.

| Check | What it validates | Failure consequence |
| --- | --- | --- |
| H1 | `tourism_activity` base values (`total_travelers`, `population_total`, `traveler_value_status`, `comparable_to_2019`) match Gold | BLOCK: base values corrupted |
| H2 | `tourism_recovery` base values match Gold for both baseline and comparison years | BLOCK: recovery inputs wrong |
| H3 | `tourism_economy` base values match Gold for 2019 and 2023 | BLOCK: economy inputs wrong |
| H4 | `traveler_segment_recovery` overall recovery index matches `tourism_recovery` recovery index | BLOCK: tables inconsistent |

**Input:** Temp views `v_tourism_activity`, `v_tourism_recovery`, `v_traveler_segment_recovery`, `v_tourism_economy` and Gold `fact_province_year`.

**Output:** Temp view `v_val_h_cross`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_h_cross AS
WITH checks AS (
  SELECT 'H1_activity_vs_gold' AS check_id, COUNT(*) AS failures FROM (
    SELECT a.province_psgc, a.year
    FROM v_tourism_activity a
    JOIN `tuklas_dev`.`04_gold`.`fact_province_year` f ON a.province_psgc = f.province_psgc AND a.year = f.year
    WHERE NOT (a.total_travelers <=> f.total_travelers
           AND a.population_total <=> f.population_total
           AND a.traveler_value_status <=> f.traveler_value_status
           AND a.comparable_to_2019 <=> f.comparable_to_2019)
  )
  UNION ALL
  SELECT 'H2_recovery_vs_gold', COUNT(*) FROM (
    SELECT r.province_psgc, r.comparison_year
    FROM v_tourism_recovery r
    JOIN `tuklas_dev`.`04_gold`.`fact_province_year` b ON r.province_psgc = b.province_psgc AND b.year = 2019
    JOIN `tuklas_dev`.`04_gold`.`fact_province_year` c ON r.province_psgc = c.province_psgc AND c.year = r.comparison_year
    WHERE NOT (r.travelers_2019 <=> b.total_travelers
           AND r.travelers_comparison_year <=> c.total_travelers
           AND r.comparable_to_2019 <=> c.comparable_to_2019)
  )
  UNION ALL
  SELECT 'H3_economy_vs_gold', COUNT(*) FROM (
    SELECT e.province_psgc
    FROM v_tourism_economy e
    JOIN `tuklas_dev`.`04_gold`.`fact_province_year` b ON e.province_psgc = b.province_psgc AND b.year = 2019
    JOIN `tuklas_dev`.`04_gold`.`fact_province_year` c ON e.province_psgc = c.province_psgc AND c.year = 2023
    WHERE NOT (e.travelers_2019 <=> b.total_travelers
           AND e.travelers_2023 <=> c.total_travelers
           AND e.afs_gva_constant_2019 <=> b.afs_gva_constant
           AND e.afs_gva_constant_2023 <=> c.afs_gva_constant
           AND e.afs_gva_current_2023 <=> c.afs_gva_current
           AND e.gdp_total_current_2023 <=> c.gdp_total_current
           AND e.gdp_per_capita_current_2023 <=> c.gdp_per_capita_current
           AND e.gdp_per_capita_current_2019 <=> b.gdp_per_capita_current)
  )
  UNION ALL
  SELECT 'H4_segment_vs_recovery', COUNT(*) FROM (
    SELECT s.province_psgc, s.comparison_year
    FROM v_traveler_segment_recovery s
    JOIN v_tourism_recovery r ON s.province_psgc = r.province_psgc AND s.comparison_year = r.comparison_year
    WHERE NOT (s.overall_recovery_index <=> r.recovery_index)
  )
  UNION ALL
  SELECT 'H5_economy_vs_recovery_2023', COUNT(*) FROM (
    SELECT e.province_psgc
    FROM v_tourism_economy e
    JOIN v_tourism_recovery r ON e.province_psgc = r.province_psgc AND r.comparison_year = 2023
    WHERE NOT (e.recovery_index_2023 <=> r.recovery_index)
       OR NOT (e.recovery_class_2023 <=> r.recovery_class)
  )
  UNION ALL
  SELECT 'H6_economy_vs_activity_intensity', COUNT(*) FROM (
    SELECT e.province_psgc
    FROM v_tourism_economy e
    JOIN v_tourism_activity a ON e.province_psgc = a.province_psgc AND a.year = 2023
    WHERE NOT (e.tourism_intensity_per_1000_2023 <=> a.tourism_intensity_per_1000)
  )
  UNION ALL
  SELECT 'H7_segment_baseline_vs_gold', COUNT(*) FROM (
    SELECT s.province_psgc, s.comparison_year
    FROM v_traveler_segment_recovery s
    JOIN `tuklas_dev`.`04_gold`.`fact_province_year` b ON s.province_psgc = b.province_psgc AND b.year = 2019
    WHERE NOT (s.domestic_travelers_2019 <=> b.domestic_travelers)
       OR NOT (s.foreign_travelers_2019 <=> b.foreign_travelers)
       OR NOT (s.overseas_filipinos_2019 <=> b.overseas_filipinos)
  )
  UNION ALL
  SELECT 'H8_segment_comparison_vs_gold', COUNT(*) FROM (
    SELECT s.province_psgc, s.comparison_year
    FROM v_traveler_segment_recovery s
    JOIN `tuklas_dev`.`04_gold`.`fact_province_year` c ON s.province_psgc = c.province_psgc AND c.year = s.comparison_year
    WHERE NOT (s.domestic_travelers_comparison <=> c.domestic_travelers)
       OR NOT (s.foreign_travelers_comparison <=> c.foreign_travelers)
       OR NOT (s.overseas_filipinos_comparison <=> c.overseas_filipinos)
  )
)
SELECT * FROM checks

### H2: Assert cross-table checks

This cell collects the cross-table check results. If any check has failures > 0, it raises an error.

**Input:** Temp view `v_val_h_cross`.

**Failure consequence:** BLOCK: cross-table inconsistency means one or more Analytics tables are wrong. The error message identifies which check failed.

In [0]:
%python
for row in spark.table("v_val_h_cross").collect():
    val_check(row.check_id, "cross-table", row.failures,
              f"{row.check_id}: {row.failures} failures", phase="Phase H")
assert_blocking("Phase H -- cross-table validation")
persist_dq_results()

### H3: Manual spot checks

This cell displays rows from the Analytics temp views for specific provinces and edge cases. These are manual checks: they do not raise errors but allow a human reviewer to verify that the values look correct.

**Spot checks include:**
- **Quezon**: province with reported values in all years.
- **Aklan**: comparability FALSE from 2021 (OI-4: Boracay reporting change). Recovery should be ineligible for 2022 and 2023.
- **Pampanga**: province with `dash_as_zero` status (overseas Filipinos reported as 0).
- **City of Angeles**: separately reported HUC.
- **Basilan**: unit with missing DOT values in 2019 (`not_reported`). Recovery should be ineligible.
- **Aklan 2022**: unit/year with `comparable_to_2019` = FALSE.

**Input:** Temp views `v_tourism_activity`, `v_tourism_recovery`, `v_traveler_segment_recovery`, `v_tourism_economy`.

**Output:** Displayed rows for manual review.

In [0]:
%sql
-- Tourism activity spot checks
SELECT 'tourism_activity' AS source_table, province_name, unit_type, year,
  total_travelers, population_total, tourism_intensity_per_1000,
  tourism_rank, intensity_rank, tourism_share_pct,
  traveler_value_status, comparable_to_2019, activity_eligible, intensity_eligible
FROM v_tourism_activity
WHERE province_name IN ('Quezon', 'Aklan', 'Pampanga', 'City of Angeles', 'Basilan')
  AND year IN (2019, 2022, 2023)
ORDER BY province_name, year

### H4: Recovery spot checks

This cell displays recovery and segment recovery rows for the spot-check provinces. It verifies that:
- Aklan's recovery is ineligible due to `comparable_to_2019` = FALSE.
- Basilan's recovery is ineligible due to NULL baseline travelers.
- Quezon and Pampanga have eligible recovery with calculated indices.
- The `below_2019_total_flag` is correctly set.

**Input:** Temp views `v_tourism_recovery` and `v_traveler_segment_recovery`.

**Output:** Displayed rows for manual review.

In [0]:
%sql
-- Recovery spot checks
SELECT 'tourism_recovery' AS source_table, province_name, comparison_year,
  travelers_2019, travelers_comparison_year, traveler_change_absolute,
  recovery_index, recovery_class, comparable_to_2019,
  recovery_eligible, recovery_exclusion_reason
FROM v_tourism_recovery
WHERE province_name IN ('Quezon', 'Aklan', 'Pampanga', 'City of Angeles', 'Basilan')
ORDER BY province_name, comparison_year

### H5: Segment recovery spot checks

This cell displays traveler-segment recovery rows for the spot-check provinces. It verifies that:
- Segment recovery indices are calculated independently for domestic, foreign, and overseas Filipino travelers.
- Ineligible segments have NULL metrics and an exclusion reason.
- The `below_2019_total_flag` and `segment_analysis_cohort_flag` are correctly set.

**Input:** Temp view `v_traveler_segment_recovery`.

**Output:** Displayed rows for manual review.

In [0]:
%sql
-- Segment recovery spot checks
SELECT 'traveler_segment_recovery' AS source_table, province_name, comparison_year,
  overall_recovery_index, overall_recovery_class,
  domestic_recovery_index, domestic_recovery_class, domestic_eligible,
  foreign_recovery_index, foreign_recovery_class, foreign_eligible,
  overseas_filipino_recovery_index, overseas_filipino_recovery_class, overseas_filipino_eligible,
  below_2019_total_flag, segment_analysis_cohort_flag
FROM v_traveler_segment_recovery
WHERE province_name IN ('Quezon', 'Aklan', 'Pampanga', 'City of Angeles', 'Basilan')
ORDER BY province_name, comparison_year

### H6: Economy spot checks

This cell displays tourism-economy rows for the spot-check provinces. It verifies that:
- `tourism_change_pct` and `afs_gva_growth_pct` are calculated correctly.
- `tourism_direction` and `economy_direction` are correct.
- `tourism_economy_pattern` is classified correctly.
- `tourism_economy_pattern_label` uses the approved wording: "Tourism activity <direction>; real A&F GVA <direction>".
- Tourism activity means reported overnight travelers, as defined by DOT.
- `afs_share_of_gdp_pct` uses current-price values.
- `gdp_per_capita_current_2019` is exposed for baseline context.

**Input:** Temp view `v_tourism_economy`.

**Output:** Displayed rows for manual review.

In [0]:
%sql
-- Economy spot checks
SELECT 'tourism_economy' AS source_table, province_name, unit_type,
  travelers_2019, travelers_2023, recovery_index_2023, recovery_class_2023,
  tourism_change_pct, afs_gva_constant_2019, afs_gva_constant_2023, afs_gva_growth_pct,
  tourism_direction, economy_direction, tourism_economy_pattern, tourism_economy_pattern_label,
  afs_gva_current_2023, gdp_total_current_2023, afs_share_of_gdp_pct,
  gdp_per_capita_current_2023, gdp_per_capita_current_2019,
  tourism_intensity_per_1000_2023
FROM v_tourism_economy
WHERE province_name IN ('Quezon', 'Aklan', 'Pampanga', 'City of Angeles', 'Basilan')
ORDER BY province_name

### H7: Rerun verification

This cell compares natural keys and business columns between the newly built temp views and the previously persisted Analytics tables. It uses bidirectional `exceptAll` (equivalent to `EXCEPT ALL`) to detect missing keys, added keys, and changed business values. Run identifiers and execution timestamps are excluded from comparison (none are stored in the Analytics tables).

**First execution:** Reports "no previous output available" rather than claiming verification passed.

**Changed Gold inputs:** Expected result changes from Gold input changes are not treated as idempotency failures.

**Supplementary:** Row count summaries are provided alongside the bidirectional comparison.

In [0]:
# Rerun verification: bidirectional comparison of natural keys and business columns
# First execution: reports 'no previous output available'
# Changed Gold inputs: expected result changes are not idempotency failures

rerun_tables = {
    'tourism_activity': {
        'view': 'v_tourism_activity',
        'keys': ['province_psgc', 'year'],
        'business_cols': ['total_travelers', 'population_total', 'population_source',
                          'tourism_intensity_per_1000', 'tourism_rank', 'intensity_rank',
                          'tourism_share_pct', 'traveler_value_status', 'comparable_to_2019',
                          'activity_eligible', 'intensity_eligible']
    },
    'tourism_recovery': {
        'view': 'v_tourism_recovery',
        'keys': ['province_psgc', 'comparison_year'],
        'business_cols': ['travelers_2019', 'travelers_comparison_year', 'recovery_index',
                          'recovery_class', 'recovery_eligible', 'comparable_to_2019']
    },
    'tourism_economy': {
        'view': 'v_tourism_economy',
        'keys': ['province_psgc'],
        'business_cols': ['recovery_index_2023', 'recovery_class_2023', 'tourism_change_pct',
                          'afs_gva_growth_pct', 'tourism_direction', 'economy_direction',
                          'tourism_economy_pattern', 'tourism_economy_pattern_label', 'alignment_eligible']
    }
}

print('RERUN VERIFICATION')
print('-' * 40)
for tbl, cfg in rerun_tables.items():
    try:
        old_df = spark.table(f'`tuklas_dev`.`05_analytics`.`{tbl}`')
    except Exception:
        print(f'  {tbl}: no previous output available (first execution)')
        continue
    new_df = spark.table(cfg['view'])
    cols = cfg['keys'] + cfg['business_cols']
    common_cols = [c for c in cols if c in old_df.columns and c in new_df.columns]
    old_sel = old_df.select(*common_cols)
    new_sel = new_df.select(*common_cols)
    missing = old_sel.exceptAll(new_sel).count()
    added = new_sel.exceptAll(old_sel).count()
    if missing == 0 and added == 0:
        print(f'  {tbl}: MATCH -- no changes from previous run')
    else:
        print(f'  {tbl}: DIFFERENCES -- {missing} missing, {added} added/changed')
    print(f'    old: {old_df.count()} rows, new: {new_df.count()} rows')

print()
print('Gold input version snapshot:')
print(f'  fact_province_year: {gold_fact_version}')
print(f'  dim_province: {gold_dim_province_version}')
print(f'  dim_year: {gold_dim_year_version}')
if gold_fact_version is not None:
    print('  If Gold versions changed since the previous run, result differences are expected.')

## Phase I -- Execution summary

**Purpose:** Print a summary based on actual evidence from this run. No hardcoded success statements or assumptions.

**Contents:**
- Outputs written during this run (with row count, version, validation status).
- Outputs not attempted or blocked.
- BLOCK failures and REVIEW flags.
- Missing Gold fields.
- Unresolved decisions.
- Blocked segment analysis.
- Validation performed.
- Rerun verification status.
- BQs/AQs supported (BQ1-BQ2 mapping).

**Input:** All validation results collected in `val_results`, `run_log`, `tables_completed`, `tables_blocked`, `missing_gold_fields`, `unresolved_decisions`.

**Output:** Printed summary. No persisted table is created.

In [0]:
%python
print("=" * 80)
print("EXECUTION SUMMARY -- Analytics Build")
print(f"Run ID: {run_id}")
print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Gold fact_province_year version: {gold_fact_version}")
print("=" * 80)

print("\n1. OUTPUTS WRITTEN DURING THIS RUN")
print("-" * 40)
expected_tables = {
    'tourism_activity': 'province_psgc x year | key: (province_psgc, year) | 600 rows',
    'tourism_concentration': 'year | key: (year) | 6 rows',
    'tourism_recovery': 'province_psgc x comparison_year | key: (province_psgc, comparison_year) | 200 rows',
    'traveler_segment_recovery': 'province_psgc x comparison_year | key: (province_psgc, comparison_year) | 200 rows',
    'tourism_economy': 'province_psgc | key: (province_psgc) | 100 rows',
}
for tbl, desc in expected_tables.items():
    log_entry = next((r for r in run_log if r[0] == tbl and r[1] == 'COMPLETED'), None)
    if log_entry:
        print(f"  + {tbl}: {desc} | actual: {log_entry[2]} rows | version: {log_entry[3]} | {log_entry[5]}")
    elif tbl in tables_blocked:
        print(f"  x {tbl}: BLOCKED -- not written this run")
    else:
        print(f"  ? {tbl}: NOT ATTEMPTED -- no run_log entry")

print("\n2. BQs/AQs SUPPORTED")
print("-" * 40)
print("  BQ1: Which provinces stand out in tourism activity and recovery, and which lag?")
print("    AQ1.1: Activity, intensity, concentration -> tourism_activity, tourism_concentration")
print("    AQ1.2: Recovery relative to 2019 -> tourism_recovery")
print("    AQ1.3: Traveler-type recovery among units below 2019 -> traveler_segment_recovery")
print("  BQ2: How is recovery in tourism associated with local economic activity?")
print("    AQ2.1: Traveler change vs real A&F GVA growth -> tourism_economy")
print("    AQ2.2: Matching vs divergent directions -> tourism_economy")
print("      Labels use 'tourism activity' and 'real A&F GVA' wording.")
print("      Tourism activity means reported overnight travelers, as defined by DOT.")
print("    AQ2.3: A&F economic significance across recovery patterns -> tourism_economy")
print("  Additional context: 2023 tourism intensity vs 2023 GDP per capita -> tourism_economy")

print("\n3. FORMULAS AND ELIGIBILITY RULES")
print("-" * 40)
print("  tourism_intensity_per_1000 = total_travelers / population_total * 1000")
print("  recovery_index = travelers_comparison / travelers_2019 * 100")
print("  tourism_change_pct = (travelers_2023 - travelers_2019) / travelers_2019 * 100")
print("  afs_gva_growth_pct = (afs_gva_constant_2023 - afs_gva_constant_2019) / afs_gva_constant_2019 * 100 (CONSTANT)")
print("  afs_share_of_gdp_pct = afs_gva_current_2023 / gdp_total_current_2023 * 100 (CURRENT)")
print("  Eligibility: baseline > 0, comparison non-NULL nonnegative, comparable_to_2019 IS TRUE")
print("  Classification: <100=below_2019, =100=unchanged, >100=above_2019, ineligible=NULL")
print("  NULL/unknown reporting statuses do not qualify silently.")
print("  Missing values never replaced with zero (DL-009).")
print("  above_2019 is NOT described as 'emerging destination' (DL-006, OI-7 thresholds not set).")
print("  100 units described as reporting units, not 100 provinces (DL-010).")

print("\n4. CHECKS EXECUTED")
print("-" * 40)
pass_count = sum(1 for r in val_results if r[3] == 'PASS')
fail_count = sum(1 for r in val_results if r[3] == 'FAIL')
block_fail = sum(1 for r in val_results if r[2] == 'BLOCK' and r[3] == 'FAIL')
review_fail = sum(1 for r in val_results if r[2] == 'REVIEW' and r[3] == 'FAIL')
print(f"  Total checks: {len(val_results)} | PASS: {pass_count} | FAIL: {fail_count}")
print(f"  BLOCK failures: {block_fail} | REVIEW flags: {review_fail}")
for r in val_results:
    marker = '+' if r[3] == 'PASS' else 'x'
    sev = f" ({r[2]})" if r[2] != 'BLOCK' else ""
    print(f"  {marker} {r[0]:<20} {r[1]:<28} {r[3]:<5}{sev} | {r[5]}")

print("\n5. OUTPUTS COMPLETED OR BLOCKED")
print("-" * 40)
print(f"  Completed this run: {', '.join(tables_completed) if tables_completed else 'none'}")
print(f"  Blocked this run:   {', '.join(tables_blocked) if tables_blocked else 'none'}")
not_attempted = [t for t in expected_tables if t not in tables_completed and t not in tables_blocked]
if not_attempted:
    print(f"  Not attempted:      {', '.join(not_attempted)}")
print("  NOTE: Separate Delta table replacements are NOT one atomic transaction.")
if len(tables_completed) > 0 and len(tables_completed) < len(expected_tables):
    print("  PARTIAL COMPLETION: only some outputs were replaced. Do not declare the Analytics layer ready.")
elif not tables_completed:
    print("  NO OUTPUTS WRITTEN. Analytics layer is not ready.")

print("\n6. MISSING GOLD FIELDS")
print("-" * 40)
if missing_gold_fields:
    for f, m in missing_gold_fields:
        print(f"  - {f}: {m}")
else:
    print("  None reported.")

print("\n7. UNRESOLVED DECISIONS")
print("-" * 40)
if unresolved_decisions:
    for d, m in unresolved_decisions:
        print(f"  - {d}: {m}")
else:
    print("  None reported.")

print("\n8. BLOCKED SEGMENT ANALYSIS")
print("-" * 40)
print("  foreign/overseas Filipino separate-segment recovery: BLOCKED")
print("    Reason: foreign_includes_overseas_filipinos not available in Gold.")
print("    Non-NULL foreign values do not prove separate category comparability.")
print("    Source values preserved; metrics set to NULL; eligibility FALSE.")
print("    Domestic recovery calculated independently and is not affected.")
print("    Region II blanket statement corrected: category interpretations")
print("    are not assumed from region name alone.")

print("\n9. VALIDATION PERFORMED")
print("-" * 40)
print("  Gold input: dim_province (100 units, non-blank labels), dim_year (exact year set, all flags non-NULL),")
print("    fact_province_year (600 rows, FK integrity, required columns).")
print("  Pre-persistence: blocking checks before each CREATE OR REPLACE TABLE.")
print("  Cross-table: base values match Gold (<=>), recovery index/class match between tables,")
print("    intensity match between tourism_economy and tourism_activity (2023).")
print("  DQ results persisted to tuklas_dev.06_data_quality.dq_results.")

print("\n10. RERUN VERIFICATION")
print("-" * 40)
print("  See H7 rerun comparison cell. First execution reports 'no previous output available'.")
print("  Reruns compare natural keys and business columns bidirectionally (EXCEPT ALL).")

print("\n" + "=" * 80)
print("END OF EXECUTION SUMMARY")
print("=" * 80)